<a href="https://colab.research.google.com/github/viratjoneja/GST-Agents-and-Tools/blob/main/ITC_version9_1_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
# GST ITC RECONCILIATION v9.1
# Y K JONEJA & CO  --  Faridabad
#
# Changes from v9.0:
#   - Document classification at load time (Invoice / Credit Note / Debit Note)
#   - Invoice and CDN/DN streams split immediately after load_and_clean
#   - stage2_invoice_matching: uses invoice stream only (portal_inv, books_inv)
#   - stage3_cdn_matching: new dedicated stage
#       * Matches by |Tax Amount| + Date; invoice numbers deliberately ignored
#       * CDN_TAX_TOLERANCE = 5.0 Rs (tighter than invoice 100 Rs)
#       * CDN_DATE_TOLERANCE = 60 days (wider — CNs uploaded in later month)
#       * GSTIN-Level Match GSTINs skipped (consistent efficiency design)
#   - stage4_coverage: covers both invoice and CDN streams (renamed from stage3)
#   - Report writer: ob_all / op_all / mp_all combine both streams
#   - GSTIN Match Report (Stage 1): UNCHANGED — audit artifact
#   - All 11 output sheet names: UNCHANGED
#
# OUTPUT:
#   ITC_Reconciliation_v9_1_<timestamp>.xlsx in BASE_PATH
#
# SHEETS (unchanged from v9.0):
#   1.  Dashboard
#   2.  GSTIN Match Report
#   3.  Party-wise Reconciliation
#   4.  Query Sheet -- Books
#   5.  Query Sheet -- Portal
#   6.  Only in Books (B-A)
#   7.  Only in Portal (A-B)
#   8.  Fuzzy Match Review
#   9.  Monthly ITC Analysis
#   10. ITC Ageing
#   11. LLM Ready Summary
# ==============================================================

import os
import re
import sys
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

try:
    from rapidfuzz import fuzz
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False


# ==============================================================
# SECTION 1 -- CONFIG
# ==============================================================

VERSION = "v9.1"

CLIENT_NAME   = "VICL Haryana"
CLIENT_GSTIN  = "06AAACV..."
RETURN_PERIOD = "Apr-2026"

BASE_PATH      = r"F:\clients_gst\VICL hry\internal audit 25-26\checkpoints"
GSTR2B_FILE    = "gstr2b_updated.xlsx"
GSTR2B_SHEET   = "Sheet1"
PURCHASE_FILE  = "COMBINED_PURCHASE_All_4_Units_Till_JAN_26.xlsx"
PURCHASE_SHEET = "BillwisePurchaseRegister"

# Standard tolerance controls
AMOUNT_TOLERANCE = 100.0
TAX_TOLERANCE    = 100.0
DATE_TOLERANCE   = 30
SIMILARITY_THRESHOLD  = 80
FRAGMENT_THRESHOLD    = 60

# Short invoice safeguards
SHORT_NUMERIC_MAX_DIGITS      = 3
SHORT_NUMERIC_DATE_TOLERANCE  = 15

# Ambiguity controls
NEEDS_REVIEW_CONFIDENCE_GAP      = 5.0
AMBIGUOUS_MATCHES_ARE_UNCONFIRMED = True

# ── v9.1: Document type classification ────────────────────────
DOC_TYPE_INVOICE     = "Invoice"
DOC_TYPE_CREDIT_NOTE = "Credit Note"
DOC_TYPE_DEBIT_NOTE  = "Debit Note"

_CDN_KEYWORDS = frozenset({
    "credit note", "credit-note", "creditnote",
    "cdnr", "cdn", "isd credit note", "isd cn",
})
_DN_KEYWORDS = frozenset({
    "debit note", "debit-note", "debitnote",
    "dnr", "dn", "dbnt",
})
_INV_KEYWORDS = frozenset({
    "invoice", "b2b", "b2ba", "inv", "tax invoice",
})

_CDN_INV_RE = re.compile(
    r'(^|[/\-\s_])(CR|CDN|CN|CDNR|RD|CRN)([/\-\s_\d]|$)',
    re.IGNORECASE
)
_DN_INV_RE = re.compile(
    r'(^|[/\-\s_])(DR|DBN|DN|DBNR|DNR|DEB)([/\-\s_\d]|$)',
    re.IGNORECASE
)

# ── v9.1: CDN-specific matching tolerances ─────────────────────
# Tighter tax tolerance: credit notes are computationally exact
# amounts so Rs 100 window causes false ambiguity (Ferro Steel case)
CDN_TAX_TOLERANCE  = 5.0
# Wider date tolerance: suppliers often upload CNs in the following
# month after the original transaction date
CDN_DATE_TOLERANCE = 60

# Derived paths
GSTR2B_PATH   = os.path.join(BASE_PATH, GSTR2B_FILE)
PURCHASE_PATH = os.path.join(BASE_PATH, PURCHASE_FILE)


# ==============================================================
# SECTION 2 -- COLUMN MAPPING
# ==============================================================

PORTAL_COLS = {
    "gstin":     "GSTIN of supplier",
    "inv_no":    "Document No.",
    "inv_date":  "Document Date",
    "taxable":   "Taxable Value (₹)",
    "igst":      "IGST Tax(₹)",
    "cgst":      "CGST Tax(₹)",
    "sgst":      "SGST Tax(₹)",
    "cess":      "CESS(₹)",
    "name":      "Trade/Legal name",
    "doc_type":  "Document Type",
}

BOOKS_COLS = {
    "gstin":    "GSTIN_NO",
    "inv_no":   "BILL_NO",
    "inv_date": "BILL_DATE",
    "taxable":  "TAXABLE",
    "igst":     "IGST AMT",
    "cgst":     "CGST AMT",
    "sgst":     "SGST AMT",
    "cess":     "Cess",
    "name":     "AC_NAME",
    "unit":     "Unit",
    "itc_igst": "MODVAT IGST",
    "itc_cgst": "MODVAT CGST",
    "itc_sgst": "MODVAT SGST",
    "doc_type": None,   # EDIT: set to column name if ERP exports doc type
                        # e.g. "VOUCHER_TYPE" for Tally
                        # Leave None to infer from sign + invoice number pattern
}

_STD = {
    "gstin":    "Supplier GSTIN",
    "inv_no":   "Invoice No",
    "inv_date": "Invoice Date",
    "taxable":  "Taxable",
    "igst":     "IGST",
    "cgst":     "CGST",
    "sgst":     "SGST",
    "cess":     "Cess",
    "name":     "Supplier Name",
    "unit":     "Unit",
    "itc_igst": "ITC_IGST",
    "itc_cgst": "ITC_CGST",
    "itc_sgst": "ITC_SGST",
    "doc_type": "Doc Type",
}

_MANDATORY = ["gstin", "inv_no", "inv_date", "taxable", "igst", "cgst", "sgst"]


# ==============================================================
# SECTION 3 -- VALIDATION
# ==============================================================

def _validate(label, filepath, sheet, col_map):
    try:
        sample = pd.read_excel(filepath, sheet_name=sheet, nrows=2)
    except Exception as e:
        print(f"  ERR  Cannot open {label}: {e}")
        return False

    actual = set(sample.columns.tolist())
    ok = True

    print(f"  {label}")
    print(f"  {'Key':<12} {'Your column':<36} {'Std name':<20} Status")
    print("  " + "-" * 82)

    for key, col in col_map.items():
        std = _STD.get(key, key)
        if col is None:
            if key in _MANDATORY:
                print(f"  {key:<12} {'None':<36} {std:<20} ERR  MANDATORY")
                ok = False
            else:
                print(f"  {key:<12} {'--':<36} {std:<20} skipped optional")
        elif col in actual:
            note = ""
            if key.startswith("itc_"):
                note = "[analytical]"
            elif key == "unit":
                note = "[unit grouping]"
            elif key == "doc_type":
                note = "[classification]"
            print(f"  {key:<12} {col:<36} {std:<20} OK  {note}")
        else:
            status = "ERR  not found" if key in _MANDATORY else "WARN not found optional"
            print(f"  {key:<12} {str(col):<36} {std:<20} {status}")
            if key in _MANDATORY:
                ok = False
    return ok


def validate_columns():
    print("COLUMN MAPPING VALIDATION")
    print("=" * 86)
    p_ok = _validate("PORTAL  (GSTR-2B)", GSTR2B_PATH, GSTR2B_SHEET, PORTAL_COLS)
    print()
    b_ok = _validate("BOOKS  (Purchase Register)", PURCHASE_PATH, PURCHASE_SHEET, BOOKS_COLS)
    print()
    print("=" * 86)
    print("  ALL MANDATORY COLUMNS VERIFIED" if p_ok and b_ok else "  FIX ERRORS ABOVE before running")
    print("=" * 86)
    return p_ok and b_ok


# ==============================================================
# SECTION 4 -- RECONCILIATION ENGINE
# ==============================================================

class GSTReconcilerV91:
    STD = _STD

    def __init__(self, cfg):
        self.cfg = cfg
        self.portal_df = None
        self.books_df  = None

        # v9.1: document streams — populated in load_and_clean
        self.portal_inv = None
        self.books_inv  = None
        self.portal_cdn = None
        self.books_cdn  = None

        self.results = {}
        self.gstin_level_match_gstins = set()
        self.mismatch_gstins          = set()
        self.portal_only_gstins       = set()
        self.books_only_gstins        = set()
        self.mismatch_gstins_sorted   = []

        print("=" * 72)
        print(f" GST ITC RECONCILIATION ENGINE {VERSION}  --  Y K JONEJA & CO")
        print("=" * 72)
        print(f"  Client        : {cfg['client_name']}")
        print(f"  Period        : {cfg.get('return_period', '')}")
        print(f"  Amt tol       : +/- Rs {cfg['amount_tolerance']}")
        print(f"  Tax tol       : +/- Rs {cfg['tax_tolerance']}")
        print(f"  Date tol      : +/- {cfg['date_tolerance']} days")
        print(f"  Similarity    : >= {cfg['similarity_threshold']}%")
        print(f"  Frag thresh   : >= {cfg['fragment_threshold']}%")
        print(f"  CDN tax tol   : +/- Rs {cfg['cdn_tax_tolerance']}")
        print(f"  CDN date tol  : +/- {cfg['cdn_date_tolerance']} days")
        print("=" * 72)

    # ----------------------------------------------------------
    # Helpers (unchanged from v9.0)
    # ----------------------------------------------------------

    def _fuzzy_score(self, s1, s2):
        s1 = "" if pd.isna(s1) else str(s1)
        s2 = "" if pd.isna(s2) else str(s2)
        if FUZZY_AVAILABLE:
            return max(fuzz.ratio(s1, s2), fuzz.partial_ratio(s1, s2), fuzz.token_sort_ratio(s1, s2))
        s1, s2 = s1.lower(), s2.lower()
        if s1 == s2: return 100
        if not s1 or not s2: return 0
        common = sum(1 for c in s1 if c in s2)
        return int(common / max(len(s1), len(s2)) * 100)

    @staticmethod
    def _norm(s):
        return re.sub(r"[^\w/\-]", "", str(s).upper().strip())

    @staticmethod
    def _compact(s):
        return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

    @staticmethod
    def _strip_zero(s):
        stripped = str(s).lstrip("0")
        return stripped if stripped else "0"

    @staticmethod
    def _safe_date_diff(d1, d2):
        try:
            if pd.isna(d1) or pd.isna(d2): return 999
            return abs((d1 - d2).days)
        except Exception:
            return 999

    @staticmethod
    def _fmt_diff(val):
        try:
            val = float(val)
        except Exception:
            return ""
        if abs(val) < 0.01: return "0.00 ="
        if val > 0: return f"+{val:,.2f} up"
        return f"{val:,.2f} dn"

    @staticmethod
    def _is_short_numeric_invoice(inv, max_digits=3):
        inv = str(inv).strip()
        return bool(re.fullmatch(rf"0*\d{{1,{max_digits}}}", inv))

    @staticmethod
    def _numeric_tokens(inv_str):
        raw = re.findall(r"\d+", str(inv_str).upper())
        filtered = set()
        for token in raw:
            n = int(token)
            if len(token) == 1: continue
            if len(token) == 2 and 20 <= n <= 30: continue
            if len(token) == 4 and 2000 <= n <= 2099: continue
            filtered.add(str(n))
        return filtered

    @staticmethod
    def _invoice_features(inv):
        raw = "" if pd.isna(inv) else str(inv).upper().strip()
        tokens = re.split(r"[^A-Z0-9]+", raw)
        tokens = [t for t in tokens if t]
        compact = re.sub(r"[^A-Z0-9]", "", raw)
        num_tokens, alpha_tokens = [], []
        for token in tokens:
            for n in re.findall(r"\d+", token):
                num_tokens.append(GSTReconcilerV91._strip_zero(n))
            for a in re.findall(r"[A-Z]+", token):
                alpha_tokens.append(a)
        return {
            "raw": raw, "compact": compact, "tokens": tokens,
            "num_tokens": num_tokens, "alpha_tokens": alpha_tokens,
            "last_num": num_tokens[-1] if num_tokens else "",
            "meaningful_numeric_tokens": sorted(GSTReconcilerV91._numeric_tokens(raw)),
        }

    def _fragment_score(self, inv_b, inv_p):
        norm_b = re.sub(r"[^\w]", "", str(inv_b).upper())
        norm_p = re.sub(r"[^\w]", "", str(inv_p).upper())
        tokens_b = self._numeric_tokens(inv_b)
        tokens_p = self._numeric_tokens(inv_p)
        if tokens_b and tokens_p:
            token_score = len(tokens_b & tokens_p) / len(tokens_b | tokens_p) * 100
        elif not tokens_b and not tokens_p:
            token_score = 50
        else:
            token_score = 0
        a, b = norm_b, norm_p
        best = 0
        for i in range(len(a)):
            for j in range(len(b)):
                k = 0
                while i + k < len(a) and j + k < len(b) and a[i+k] == b[j+k]:
                    k += 1
                best = max(best, k)
        lcs_score = best / max(len(a), len(b), 1) * 100
        return round(token_score * 0.70 + lcs_score * 0.30, 1)

    def _smart_invoice_score(self, b_row, p_row):
        b_compact = str(b_row.get("_inv_compact", ""))
        p_compact = str(p_row.get("_inv_compact", ""))
        b_last  = str(b_row.get("_inv_last_num", ""))
        p_last  = str(p_row.get("_inv_last_num", ""))
        b_nums  = set(b_row.get("_inv_num_tokens", []) or [])
        p_nums  = set(p_row.get("_inv_num_tokens", []) or [])
        b_alpha = set(b_row.get("_inv_alpha_tokens", []) or [])
        p_alpha = set(p_row.get("_inv_alpha_tokens", []) or [])

        if b_compact and p_compact and b_compact == p_compact:
            return 100.0, "Compact invoice number exact match"
        if b_last and p_last and b_last == p_last:
            if len(b_nums) == 1 or len(p_nums) == 1:
                return 94.0, "Last numeric token / suffix match"
            return 88.0, "Last numeric token match with multiple numeric tokens"
        if b_compact and p_compact and min(len(b_compact), len(p_compact)) >= 4:
            if b_compact in p_compact or p_compact in b_compact:
                return 86.0, "Compact invoice substring match"
        if b_nums and p_nums:
            overlap = b_nums & p_nums
            if overlap:
                ratio = len(overlap) / max(len(b_nums), len(p_nums))
                if ratio >= 0.75: return 84.0, "Strong numeric token overlap"
                if ratio >= 0.50: return 76.0, "Partial numeric token overlap"
        alpha_support = 5.0 if b_alpha and p_alpha and b_alpha & p_alpha else 0.0
        sim = float(self._fuzzy_score(b_compact, p_compact))
        return min(100.0, sim + alpha_support), "Fuzzy compact similarity"

    def _confidence_score(self, sim, amt_diff, tax_diff, date_diff, b_row):
        taxable_base = max(abs(float(b_row.get("Taxable", 0))), 1.0)
        tax_base     = max(abs(float(b_row.get("Total Tax", 0))), 1.0)
        amount_score = max(0.0, 1.0 - float(amt_diff) / taxable_base)
        tax_score    = max(0.0, 1.0 - float(tax_diff) / tax_base)
        date_score   = max(0.0, 1.0 - float(date_diff) / 90.0)
        return round(
            ((sim / 100.0) * 0.45 + amount_score * 0.25 +
             tax_score * 0.20 + date_score * 0.10) * 100, 2
        )

    def _apply_mapping(self, raw_df, col_map, label):
        rename = {}
        for key, raw_col in col_map.items():
            if raw_col is None: continue
            if raw_col not in raw_df.columns: continue
            rename[raw_col] = self.STD.get(key, key)
        mapped = raw_df[list(rename.keys())].rename(columns=rename).copy()
        print(f"  {label:<8}: {len(mapped):,} rows, {len(rename)} columns mapped")
        return mapped

    # ----------------------------------------------------------
    # v9.1: Document type classification
    # ----------------------------------------------------------

    def _classify_doc_type(self, row, side):
        """
        Classify a document row as Invoice / Credit Note / Debit Note.

        Three-layer priority:
          Layer 1  Explicit Doc Type column   Most reliable when present.
                                              Portal: always from GSTR-2B.
                                              Books:  from ERP if doc_type mapped.
          Layer 2  Sign of Total Tax          Reliable for credit notes only.
                                              Negative tax → Credit Note.
          Layer 3  Invoice number pattern     Fallback for CR/CDN/DN prefixes.

        Debit Notes are positive amounts — cannot be distinguished from
        invoices by sign alone. They require Layer 1 or Layer 3.
        On the portal side GSTR-2B always provides Doc Type so Layer 1
        handles debit notes reliably there.
        """
        # Layer 1
        doc_col = "Doc Type"
        if doc_col in row.index and pd.notna(row[doc_col]) and str(row[doc_col]).strip():
            dt = str(row[doc_col]).strip().lower()
            if dt in _CDN_KEYWORDS: return DOC_TYPE_CREDIT_NOTE
            if dt in _DN_KEYWORDS:  return DOC_TYPE_DEBIT_NOTE
            if dt in _INV_KEYWORDS: return DOC_TYPE_INVOICE

        # Layer 2 — sign
        try:
            tax = float(row.get("Total Tax", 0))
        except (TypeError, ValueError):
            tax = 0.0
        if tax < -0.01:
            return DOC_TYPE_CREDIT_NOTE

        # Layer 3 — invoice number prefix
        inv_no = str(row.get("Invoice No", ""))
        if _CDN_INV_RE.search(inv_no): return DOC_TYPE_CREDIT_NOTE
        if _DN_INV_RE.search(inv_no):  return DOC_TYPE_DEBIT_NOTE

        return DOC_TYPE_INVOICE

    # ----------------------------------------------------------
    # v9.1: CDN confidence scoring
    # ----------------------------------------------------------

    def _cdn_confidence(self, tax_diff, amt_diff, date_diff):
        """
        Confidence score for a CDN amount+date match.

        Starts at 95: amount match is the primary signal.
        Penalised by tax diff, date diff, taxable diff.
        """
        score = 95.0
        score -= min(tax_diff  * 2.00, 10.0)
        score -= min(date_diff * 0.25, 15.0)
        score -= min(amt_diff  * 0.005, 5.0)
        return round(max(score, 40.0), 2)

    # ----------------------------------------------------------
    # Pair builder (unchanged from v9.0)
    # ----------------------------------------------------------

    def _build_pair(self, gstin, b_row, p_row, amt_diff, tax_diff,
                    date_diff, sim, conf, match_type, match_reason, is_confirmed):
        taxable_diff   = round(float(b_row["Taxable"])   - float(p_row["Taxable"]),   2)
        tax_total_diff = round(float(b_row["Total Tax"]) - float(p_row["Total Tax"]), 2)
        return {
            "Supplier GSTIN":  gstin,
            "Supplier Name":   b_row.get("Supplier Name", ""),
            "Unit":            b_row.get("Unit", ""),
            "Books Invoice":   b_row.get("Invoice No", ""),
            "Books Date":      b_row.get("Invoice Date", ""),
            "Books Taxable":   b_row.get("Taxable", 0),
            "Books Tax":       b_row.get("Total Tax", 0),
            "Portal Invoice":  p_row.get("Invoice No", ""),
            "Portal Date":     p_row.get("Invoice Date", ""),
            "Portal Taxable":  p_row.get("Taxable", 0),
            "Portal Tax":      p_row.get("Total Tax", 0),
            "Taxable Diff":        taxable_diff,
            "Taxable Diff Fmt":    self._fmt_diff(taxable_diff),
            "Tax Diff":            tax_total_diff,
            "Tax Diff Fmt":        self._fmt_diff(tax_total_diff),
            "Amt Diff":            round(float(amt_diff),  2),
            "Tax Abs Diff":        round(float(tax_diff),  2),
            "Date Diff Days":      date_diff,
            "Similarity":          round(float(sim),  1),
            "Confidence":          round(float(conf), 2),
            "Match_Type":          match_type,
            "Match_Reason":        match_reason,
            "Is_Confirmed":        bool(is_confirmed),
            "Books_TXN_ID":        b_row["TXN_ID"],
            "Portal_TXN_ID":       p_row["TXN_ID"],
        }

    def _short_invoice_safety_failed(self, b_row, p_row, amt_diff, tax_diff, date_diff):
        max_digits    = self.cfg["short_numeric_max_digits"]
        short_numeric = (
            self._is_short_numeric_invoice(b_row.get("Invoice No", ""), max_digits) or
            self._is_short_numeric_invoice(p_row.get("Invoice No", ""), max_digits)
        )
        if not short_numeric: return False
        if amt_diff  > self.cfg["amount_tolerance"]:             return True
        if tax_diff  > self.cfg["tax_tolerance"]:                return True
        if date_diff > self.cfg["short_numeric_date_tolerance"]: return True
        return False

    # ----------------------------------------------------------
    # v9.1: CDN pair matching
    # ----------------------------------------------------------

    def _match_cdn_pairs(self, b_idx_list, p_idx_list, gstin):
        """
        Match CDN/DN rows by |Tax Amount| + Date proximity.
        Invoice numbers are deliberately ignored.

        Single candidate   → CDN_Amount_Match
                             Is_Confirmed = conf >= similarity_threshold
        Multiple candidates:
          gap >= NEEDS_REVIEW_CONFIDENCE_GAP → CDN_Amount_Match, Is_Confirmed=False
          gap <  NEEDS_REVIEW_CONFIDENCE_GAP → CDN_Ambiguous,    Is_Confirmed=False
        No candidates → stays in exception bucket (only-in-books/portal)
        """
        tax_tol  = self.cfg.get("cdn_tax_tolerance",  CDN_TAX_TOLERANCE)
        date_tol = self.cfg.get("cdn_date_tolerance", CDN_DATE_TOLERANCE)
        gap_tol  = self.cfg.get("needs_review_confidence_gap", NEEDS_REVIEW_CONFIDENCE_GAP)
        sim_thr  = self.cfg.get("similarity_threshold", SIMILARITY_THRESHOLD)

        b_df = self.books_df.loc[b_idx_list]
        p_df = self.portal_df.loc[p_idx_list]

        pairs  = []
        used_b = set()
        used_p = set()

        for b_idx, b_row in b_df.iterrows():
            if b_idx in used_b:
                continue

            b_tax  = abs(float(b_row.get("Total Tax", 0)))
            b_amt  = abs(float(b_row.get("Taxable",   0)))
            b_date = b_row.get("Invoice Date")
            candidates = []

            for p_idx, p_row in p_df.iterrows():
                if p_idx in used_p:
                    continue
                p_tax  = abs(float(p_row.get("Total Tax", 0)))
                p_amt  = abs(float(p_row.get("Taxable",   0)))
                p_date = p_row.get("Invoice Date")

                tax_diff  = abs(b_tax - p_tax)
                amt_diff  = abs(b_amt - p_amt)
                date_diff = self._safe_date_diff(b_date, p_date)

                if tax_diff  > tax_tol:  continue
                if date_diff > date_tol: continue

                conf = self._cdn_confidence(tax_diff, amt_diff, date_diff)
                candidates.append((p_idx, p_row, tax_diff, amt_diff, date_diff, conf))

            if not candidates:
                continue

            candidates.sort(key=lambda x: x[5], reverse=True)
            p_idx, p_row, tax_diff, amt_diff, date_diff, conf = candidates[0]

            if len(candidates) == 1:
                match_type   = "CDN_Amount_Match"
                is_confirmed = conf >= sim_thr
                reason = (
                    f"CDN matched by |tax|+date — invoice numbers ignored. "
                    f"Books:[{b_row['Invoice No']}] "
                    f"Portal:[{p_row['Invoice No']}] "
                    f"|TaxDiff|:{tax_diff:.2f} DateDiff:{date_diff}d"
                )
            else:
                gap          = conf - candidates[1][5]
                is_confirmed = False
                match_type   = "CDN_Amount_Match" if gap >= gap_tol else "CDN_Ambiguous"
                conf         = round(conf * (0.85 if gap >= gap_tol else 0.70), 2)
                reason = (
                    f"CDN {'best match' if gap >= gap_tol else 'AMBIGUOUS — review'}. "
                    f"Books:[{b_row['Invoice No']}] "
                    f"Best:[{p_row['Invoice No']}] conf={conf:.1f} "
                    f"Runner-up:[{candidates[1][1]['Invoice No']}] "
                    f"conf={candidates[1][5]:.1f} gap={gap:.1f}"
                )

            pairs.append(self._build_pair(
                gstin, b_row, p_row,
                amt_diff, tax_diff, date_diff,
                0,     # similarity = 0; invoice# not used
                conf, match_type, reason, is_confirmed
            ))
            used_b.add(b_idx)
            used_p.add(p_idx)

        remaining_b = [i for i in b_idx_list if i not in used_b]
        remaining_p = [i for i in p_idx_list if i not in used_p]
        return pairs, remaining_b, remaining_p

    # ----------------------------------------------------------
    # Stage 0 — Load and Clean
    # ----------------------------------------------------------

    def load_and_clean(self):
        print("\nSTAGE 0 : Loading and Cleaning Data")
        print("-" * 72)
        cfg = self.cfg

        try:
            raw_p = pd.read_excel(cfg["gstr2b_path"], sheet_name=cfg["gstr2b_sheet"])
        except Exception as e:
            print(f"  ERR portal : {e}")
            return False
        self.portal_df = self._apply_mapping(raw_p, cfg["portal_cols"], "Portal")

        try:
            raw_b = pd.read_excel(cfg["purchase_path"], sheet_name=cfg["purchase_sheet"])
        except Exception as e:
            print(f"  ERR books  : {e}")
            return False
        self.books_df = self._apply_mapping(raw_b, cfg["books_cols"], "Books")

        for label, df in [("Portal", self.portal_df), ("Books", self.books_df)]:
            for optional_col in ["Supplier Name", "Unit", "Doc Type"]:
                if optional_col not in df.columns:
                    df[optional_col] = ""

            df["Supplier GSTIN"] = df["Supplier GSTIN"].fillna("UNKNOWN").astype(str).str.upper().str.strip()
            df["Invoice No"]     = df["Invoice No"].fillna("").astype(str).str.upper().str.strip()
            df["_inv_norm"]      = df["Invoice No"].apply(self._norm)
            df["_inv_compact"]   = df["Invoice No"].apply(self._compact)

            features = df["Invoice No"].apply(self._invoice_features)
            df["_inv_tokens"]          = features.apply(lambda x: x["tokens"])
            df["_inv_num_tokens"]      = features.apply(lambda x: x["num_tokens"])
            df["_inv_alpha_tokens"]    = features.apply(lambda x: x["alpha_tokens"])
            df["_inv_last_num"]        = features.apply(lambda x: x["last_num"])
            df["_inv_meaningful_nums"] = features.apply(lambda x: x["meaningful_numeric_tokens"])

            df["Invoice Date"] = pd.to_datetime(df["Invoice Date"], errors="coerce")
            for col in ["Taxable", "IGST", "CGST", "SGST", "Cess"]:
                if col not in df.columns:
                    df[col] = 0.0
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(2)
            df["Total Tax"] = (df["IGST"] + df["CGST"] + df["SGST"] + df["Cess"]).round(2)

            for col in ["ITC_IGST", "ITC_CGST", "ITC_SGST"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(2)

            print(f"  {label:<8}: invoice features added")

        self.portal_df = self.portal_df.reset_index(drop=True)
        self.books_df  = self.books_df.reset_index(drop=True)
        self.portal_df["TXN_ID"] = [f"P_{i}" for i in self.portal_df.index]
        self.books_df["TXN_ID"]  = [f"B_{i}" for i in self.books_df.index]

        if "ITC_IGST" in self.books_df.columns:
            gross = self.books_df["IGST"].sum()
            itc   = self.books_df["ITC_IGST"].sum()
            print("\n  ITC NOTE (Books IGST):")
            print(f"    Gross IGST on invoices : Rs {gross:>15,.2f}")
            print(f"    ITC availed (MODVAT)   : Rs {itc:>15,.2f}")
            print(f"    Blocked / ineligible   : Rs {gross - itc:>15,.2f}")

        print(f"\n  TXN IDs : P_0--P_{len(self.portal_df)-1} | B_0--B_{len(self.books_df)-1}")

        # ── v9.1: Document classification and stream splitting ─────
        print("\n  Document Classification:")
        for label, df, side in [
            ("Portal", self.portal_df, "portal"),
            ("Books",  self.books_df,  "books"),
        ]:
            df["Doc_Class"] = df.apply(
                lambda r: self._classify_doc_type(r, side), axis=1
            )
            counts = df["Doc_Class"].value_counts()
            print(f"  {label}:")
            for dt in [DOC_TYPE_INVOICE, DOC_TYPE_CREDIT_NOTE, DOC_TYPE_DEBIT_NOTE]:
                print(f"    {dt:<16}: {counts.get(dt, 0):>6,}")

        self.portal_inv = self.portal_df[
            self.portal_df["Doc_Class"] == DOC_TYPE_INVOICE].copy()
        self.books_inv  = self.books_df[
            self.books_df["Doc_Class"] == DOC_TYPE_INVOICE].copy()
        self.portal_cdn = self.portal_df[
            self.portal_df["Doc_Class"] != DOC_TYPE_INVOICE].copy()
        self.books_cdn  = self.books_df[
            self.books_df["Doc_Class"] != DOC_TYPE_INVOICE].copy()

        print(f"\n  Invoice stream : Portal {len(self.portal_inv):,}  Books {len(self.books_inv):,}")
        print(f"  CDN/DN stream  : Portal {len(self.portal_cdn):,}  Books {len(self.books_cdn):,}")
        return True

    # ----------------------------------------------------------
    # Stage 1 -- GSTIN-Level Aggregation (UNCHANGED from v9.0)
    # Uses portal_df and books_df — all documents — audit artifact
    # ----------------------------------------------------------

    def stage1_gstin(self):
        print("\nSTAGE 1 : GSTIN-Level Aggregation")
        print("-" * 72)

        p_agg = self.portal_df.groupby("Supplier GSTIN").agg(
            Portal_Count=("Invoice No",    "count"),
            Portal_Taxable=("Taxable",     "sum"),
            Portal_IGST=("IGST",           "sum"),
            Portal_CGST=("CGST",           "sum"),
            Portal_SGST=("SGST",           "sum"),
            Portal_Tax=("Total Tax",       "sum"),
            Portal_Name=("Supplier Name",  "first"),
        ).reset_index()

        b_agg = self.books_df.groupby("Supplier GSTIN").agg(
            Books_Count=("Invoice No",     "count"),
            Books_Taxable=("Taxable",      "sum"),
            Books_IGST=("IGST",            "sum"),
            Books_CGST=("CGST",            "sum"),
            Books_SGST=("SGST",            "sum"),
            Books_Tax=("Total Tax",        "sum"),
            Books_Name=("Supplier Name",   "first"),
        ).reset_index()

        itc_cols = [c for c in ["ITC_IGST", "ITC_CGST", "ITC_SGST"]
                    if c in self.books_df.columns]
        if itc_cols:
            b_agg = b_agg.merge(
                self.books_df.groupby("Supplier GSTIN")[itc_cols].sum().reset_index(),
                on="Supplier GSTIN", how="left"
            )

        summary = p_agg.merge(b_agg, on="Supplier GSTIN", how="outer")

        for col in summary.columns:
            if col not in ["Supplier GSTIN", "Portal_Name", "Books_Name"]:
                summary[col] = pd.to_numeric(summary[col], errors="ignore")
        numeric_cols = [
            c for c in summary.columns
            if c not in ["Supplier GSTIN", "Portal_Name", "Books_Name"]
            and summary[c].dtype.kind in "biufc"
        ]
        summary[numeric_cols]  = summary[numeric_cols].fillna(0)
        summary["Portal_Name"] = summary.get("Portal_Name", "").fillna("").astype(str)
        summary["Books_Name"]  = summary.get("Books_Name",  "").fillna("").astype(str)
        summary["Supplier Name"] = summary.apply(
            lambda r: r["Books_Name"] if r["Books_Name"] else r["Portal_Name"], axis=1
        )
        summary.drop(columns=["Books_Name", "Portal_Name"], errors="ignore", inplace=True)

        summary["Diff_Count"]   = (summary["Books_Count"]   - summary["Portal_Count"]).astype(int)
        summary["Diff_Taxable"] = (summary["Books_Taxable"] - summary["Portal_Taxable"]).round(2)
        summary["Diff_Tax"]     = (summary["Books_Tax"]     - summary["Portal_Tax"]).round(2)

        amount_tol = self.cfg["amount_tolerance"]
        tax_tol    = self.cfg["tax_tolerance"]

        def classify(row):
            if row["Portal_Count"] == 0: return "Books Only"
            if row["Books_Count"]  == 0: return "Portal Only"
            if (row["Diff_Count"] == 0 and
                    abs(row["Diff_Taxable"]) <= amount_tol and
                    abs(row["Diff_Tax"])     <= tax_tol):
                return "GSTIN-Level Match"
            return "Mismatch"

        summary["Status"] = summary.apply(classify, axis=1)
        self.results["gstin_summary"] = summary

        self.gstin_level_match_gstins = set(summary.loc[summary["Status"] == "GSTIN-Level Match", "Supplier GSTIN"])
        self.mismatch_gstins          = set(summary.loc[summary["Status"] == "Mismatch",          "Supplier GSTIN"])
        self.portal_only_gstins       = set(summary.loc[summary["Status"] == "Portal Only",       "Supplier GSTIN"])
        self.books_only_gstins        = set(summary.loc[summary["Status"] == "Books Only",        "Supplier GSTIN"])

        gstin_p = int(summary.loc[summary["Status"] == "GSTIN-Level Match", "Portal_Count"].sum())
        gstin_b = int(summary.loc[summary["Status"] == "GSTIN-Level Match", "Books_Count"].sum())
        self.results["gstin_level_p_count"] = gstin_p
        self.results["gstin_level_b_count"] = gstin_b

        dist = summary["Status"].value_counts()
        print(f"  Total GSTINs       : {len(summary):,}")
        for status in ["GSTIN-Level Match", "Mismatch", "Portal Only", "Books Only"]:
            print(f"    {status:<20}: {dist.get(status, 0):>6,}")

        print("\n  Matching will run only for GSTINs classified as Mismatch.")
        print(f"    GSTIN-Level Match : {gstin_p:,} portal + {gstin_b:,} books")

        mis_rows = summary[summary["Status"] == "Mismatch"].copy()
        mis_rows["Total_Count"] = mis_rows["Portal_Count"] + mis_rows["Books_Count"]
        self.mismatch_gstins_sorted = (
            mis_rows.sort_values("Total_Count", ascending=False)["Supplier GSTIN"].tolist()
        )

    # ----------------------------------------------------------
    # Stage 2 -- Invoice Matching (v9.1: invoice stream only)
    # Pass 1 Exact → Pass 2 Fuzzy → Pass 3 Fragment unchanged
    # Only change: portal_inv / books_inv instead of portal_df / books_df
    # ----------------------------------------------------------

    def stage2_invoice_matching(self):
        print("\nSTAGE 2 : Progressive Invoice Matching  (invoice stream only)")
        print("-" * 72)

        amount_tol    = self.cfg["amount_tolerance"]
        tax_tol       = self.cfg["tax_tolerance"]
        date_tol      = self.cfg["date_tolerance"]
        sim_threshold = self.cfg["similarity_threshold"]
        frag_threshold = self.cfg["fragment_threshold"]

        matched_pairs = []
        matched_p_ids = set()
        matched_b_ids = set()

        # v9.1: invoice streams replace full dataframes
        portal_pool = self.portal_inv[
            self.portal_inv["Supplier GSTIN"].isin(self.mismatch_gstins)
        ].set_index("TXN_ID").copy()
        books_pool = self.books_inv[
            self.books_inv["Supplier GSTIN"].isin(self.mismatch_gstins)
        ].set_index("TXN_ID").copy()

        total     = len(self.mismatch_gstins_sorted)
        processed = 0
        t_start   = datetime.now()

        for gstin in self.mismatch_gstins_sorted:
            processed += 1
            p_grp = portal_pool[portal_pool["Supplier GSTIN"] == gstin].reset_index()
            b_grp = books_pool[books_pool["Supplier GSTIN"] == gstin].reset_index()

            if p_grp.empty or b_grp.empty:
                continue

            used_p = set()
            used_b = set()

            # PASS 1: Exact full invoice match
            p_dict = {}
            for _, p_row in p_grp.iterrows():
                p_dict.setdefault(p_row["_inv_norm"], []).append(p_row)

            for _, b_row in b_grp.iterrows():
                key = b_row["_inv_norm"]
                if key not in p_dict:
                    continue
                for p_row in p_dict[key]:
                    if p_row["TXN_ID"] in used_p:
                        continue
                    amt_diff  = abs(float(b_row["Taxable"])   - float(p_row["Taxable"]))
                    tax_diff  = abs(float(b_row["Total Tax"]) - float(p_row["Total Tax"]))
                    date_diff = self._safe_date_diff(b_row["Invoice Date"], p_row["Invoice Date"])
                    if amt_diff <= amount_tol and tax_diff <= tax_tol and date_diff <= date_tol:
                        pair = self._build_pair(
                            gstin, b_row, p_row, amt_diff, tax_diff, date_diff,
                            100.0, 100.0, "Exact",
                            "Normalized invoice number, taxable value, total tax and date within tolerance",
                            True,
                        )
                        matched_pairs.append(pair)
                        used_p.add(p_row["TXN_ID"])
                        used_b.add(b_row["TXN_ID"])
                        matched_p_ids.add(p_row["TXN_ID"])
                        matched_b_ids.add(b_row["TXN_ID"])
                        break

            # PASS 2: Smart/fuzzy match with tax-aware confidence
            unmatched_b = b_grp[~b_grp["TXN_ID"].isin(used_b)]
            unmatched_p = p_grp[~p_grp["TXN_ID"].isin(used_p)]

            if not unmatched_b.empty and not unmatched_p.empty:
                p_sorted     = unmatched_p.sort_values("Taxable").reset_index(drop=True)
                p_taxable    = p_sorted["Taxable"].values
                fuzzy_amt_w  = amount_tol * 5
                fuzzy_tax_w  = tax_tol    * 5
                fuzzy_date_w = date_tol   * 2

                for _, b_row in unmatched_b.iterrows():
                    if b_row["TXN_ID"] in used_b:
                        continue
                    lo_idx = np.searchsorted(p_taxable, b_row["Taxable"] - fuzzy_amt_w, side="left")
                    hi_idx = np.searchsorted(p_taxable, b_row["Taxable"] + fuzzy_amt_w, side="right")
                    candidates = p_sorted.iloc[lo_idx:hi_idx]
                    scored = []

                    for _, p_row in candidates.iterrows():
                        if p_row["TXN_ID"] in used_p:
                            continue
                        sim, reason = self._smart_invoice_score(b_row, p_row)
                        if sim < sim_threshold:
                            continue
                        amt_diff  = abs(float(b_row["Taxable"])   - float(p_row["Taxable"]))
                        tax_diff  = abs(float(b_row["Total Tax"]) - float(p_row["Total Tax"]))
                        date_diff = self._safe_date_diff(b_row["Invoice Date"], p_row["Invoice Date"])
                        if (amt_diff > fuzzy_amt_w or tax_diff > fuzzy_tax_w or
                                date_diff > fuzzy_date_w):
                            continue
                        if self._short_invoice_safety_failed(b_row, p_row, amt_diff, tax_diff, date_diff):
                            continue
                        conf = self._confidence_score(sim, amt_diff, tax_diff, date_diff, b_row)
                        scored.append((conf, sim, reason, amt_diff, tax_diff, date_diff, p_row))

                    if not scored:
                        continue
                    scored.sort(key=lambda x: x[0], reverse=True)
                    best = scored[0]
                    best_conf, best_sim, best_reason, best_amt, best_tax, best_date, best_p = best

                    ambiguous = (len(scored) > 1 and
                                 best_conf - scored[1][0] < self.cfg["needs_review_confidence_gap"])
                    is_confirmed = not (ambiguous and self.cfg["ambiguous_matches_are_unconfirmed"])
                    match_type   = "Fuzzy-Ambiguous" if ambiguous else "Fuzzy"
                    pair = self._build_pair(
                        gstin, b_row, best_p, best_amt, best_tax, best_date,
                        best_sim, best_conf, match_type, best_reason, is_confirmed,
                    )
                    matched_pairs.append(pair)
                    if is_confirmed:
                        used_p.add(best_p["TXN_ID"])
                        used_b.add(b_row["TXN_ID"])
                        matched_p_ids.add(best_p["TXN_ID"])
                        matched_b_ids.add(b_row["TXN_ID"])

            # PASS 3: Date-gated fragment match
            unmatched_b3 = b_grp[~b_grp["TXN_ID"].isin(used_b)]
            unmatched_p3 = p_grp[~p_grp["TXN_ID"].isin(used_p)]

            if not unmatched_b3.empty and not unmatched_p3.empty:
                for _, b_row in unmatched_b3.iterrows():
                    if b_row["TXN_ID"] in used_b:
                        continue
                    candidates = []
                    for _, p_row in unmatched_p3.iterrows():
                        if p_row["TXN_ID"] in used_p:
                            continue
                        amt_diff  = abs(float(b_row["Taxable"])   - float(p_row["Taxable"]))
                        tax_diff  = abs(float(b_row["Total Tax"]) - float(p_row["Total Tax"]))
                        date_diff = self._safe_date_diff(b_row["Invoice Date"], p_row["Invoice Date"])
                        if amt_diff > amount_tol or tax_diff > tax_tol or date_diff > date_tol:
                            continue
                        if self._short_invoice_safety_failed(b_row, p_row, amt_diff, tax_diff, date_diff):
                            continue
                        frag       = self._fragment_score(b_row["Invoice No"], p_row["Invoice No"])
                        smart_sim, smart_reason = self._smart_invoice_score(b_row, p_row)
                        combined   = round(max(frag, smart_sim), 1)
                        reason     = f"Fragment score {frag}; {smart_reason}"
                        candidates.append((combined, frag, smart_sim, reason,
                                           amt_diff, tax_diff, date_diff, p_row))

                    if not candidates:
                        continue
                    candidates.sort(key=lambda x: x[0], reverse=True)
                    best_score, best_frag, best_smart, best_reason, \
                        best_amt, best_tax, best_date, best_p = candidates[0]

                    close_competitor = (len(candidates) > 1 and
                                        best_score - candidates[1][0] < self.cfg["needs_review_confidence_gap"])
                    below_threshold  = best_frag < frag_threshold and best_smart < sim_threshold
                    is_confirmed     = not close_competitor and not below_threshold
                    match_type       = "Pass3-Fragment" if is_confirmed else "Pass3-Ambiguous"
                    conf = self._confidence_score(best_score, best_amt, best_tax, best_date, b_row)
                    pair = self._build_pair(
                        gstin, b_row, best_p, best_amt, best_tax, best_date,
                        best_score, conf, match_type, best_reason, is_confirmed,
                    )
                    matched_pairs.append(pair)
                    if is_confirmed:
                        used_p.add(best_p["TXN_ID"])
                        used_b.add(b_row["TXN_ID"])
                        matched_p_ids.add(best_p["TXN_ID"])
                        matched_b_ids.add(b_row["TXN_ID"])

            portal_pool = portal_pool.drop(
                index=[i for i in used_p if i in portal_pool.index], errors="ignore")
            books_pool  = books_pool.drop(
                index=[i for i in used_b if i in books_pool.index], errors="ignore")

            if processed % 10 == 0 or processed == total:
                elapsed = (datetime.now() - t_start).total_seconds()
                print(f"  {processed:>4}/{total} | pool "
                      f"{len(portal_pool):>7,}p {len(books_pool):>7,}b | "
                      f"{elapsed:.0f}s elapsed", flush=True)

        mp = pd.DataFrame(matched_pairs) if matched_pairs else pd.DataFrame()
        self.results["matched_pairs"] = mp
        self.results["matched_p_ids"] = matched_p_ids
        self.results["matched_b_ids"] = matched_b_ids

        # v9.1: use invoice streams for GSTIN-level id sets and exception buckets
        gstin_p_ids = set(self.portal_inv.loc[
            self.portal_inv["Supplier GSTIN"].isin(self.gstin_level_match_gstins),
            "TXN_ID"
        ])
        gstin_b_ids = set(self.books_inv.loc[
            self.books_inv["Supplier GSTIN"].isin(self.gstin_level_match_gstins),
            "TXN_ID"
        ])

        all_matched_p = matched_p_ids | gstin_p_ids
        all_matched_b = matched_b_ids | gstin_b_ids

        self.results["only_portal"] = self.portal_inv[
            ~self.portal_inv["TXN_ID"].isin(all_matched_p)].copy()
        self.results["only_books"] = self.books_inv[
            ~self.books_inv["TXN_ID"].isin(all_matched_b)].copy()

        self.results["only_portal"]["Action"] = (
            "Invoice in GSTR-2B but not in books — verify booking, period or invoice number.")
        self.results["only_books"]["Action"] = (
            "Invoice in books but not in GSTR-2B — verify supplier filing, period, GSTIN or ITC eligibility.")

        print("\n  Invoice-level match breakup:")
        if mp.empty:
            print("    No invoice-level match records.")
        else:
            for k, v in mp["Match_Type"].value_counts().items():
                print(f"    {k:<24}: {v:>7,}")
            print(f"    Confirmed total          : {int(mp['Is_Confirmed'].sum()):>7,}")
        print(f"  Only in Portal (invoices) : {len(self.results['only_portal']):,}")
        print(f"  Only in Books  (invoices) : {len(self.results['only_books']):,}")

    # ----------------------------------------------------------
    # Stage 3 -- CDN/DN Matching (NEW in v9.1)
    # ----------------------------------------------------------

    def stage3_cdn_matching(self):
        """
        Dedicated CDN/DN matching stage.
        Operates on portal_cdn and books_cdn streams only.
        GSTIN-Level Match GSTINs are skipped (efficiency design).
        """
        print("\nSTAGE 3 : Credit Note / Debit Note Matching  (CDN stream)")
        print("-" * 72)
        print("  Invoice numbers ignored — matching by |Tax| + Date only")
        print(f"  CDN tax tolerance  : Rs {self.cfg.get('cdn_tax_tolerance', CDN_TAX_TOLERANCE)}")
        print(f"  CDN date tolerance : {self.cfg.get('cdn_date_tolerance', CDN_DATE_TOLERANCE)} days")

        cdn_pairs  = []
        only_b_cdn = []
        only_p_cdn = []

        # Mismatch GSTINs: run CDN matching
        for gstin in self.mismatch_gstins_sorted:
            p_sub = self.portal_cdn[self.portal_cdn["Supplier GSTIN"] == gstin]
            b_sub = self.books_cdn[self.books_cdn["Supplier GSTIN"] == gstin]

            if p_sub.empty and b_sub.empty:
                continue
            if p_sub.empty:
                only_b_cdn.extend(b_sub.index.tolist())
                continue
            if b_sub.empty:
                only_p_cdn.extend(p_sub.index.tolist())
                continue

            pairs, rem_b, rem_p = self._match_cdn_pairs(
                b_sub.index.tolist(), p_sub.index.tolist(), gstin
            )
            cdn_pairs.extend(pairs)
            only_b_cdn.extend(rem_b)
            only_p_cdn.extend(rem_p)

        # Portal-Only GSTINs: all their CDNs → only_p_cdn directly
        for gstin in self.portal_only_gstins:
            p_sub = self.portal_cdn[self.portal_cdn["Supplier GSTIN"] == gstin]
            only_p_cdn.extend(p_sub.index.tolist())

        # Books-Only GSTINs: all their CDNs → only_b_cdn directly
        for gstin in self.books_only_gstins:
            b_sub = self.books_cdn[self.books_cdn["Supplier GSTIN"] == gstin]
            only_b_cdn.extend(b_sub.index.tolist())

        self.results["cdn_pairs"]  = (
            pd.DataFrame(cdn_pairs) if cdn_pairs else pd.DataFrame()
        )
        self.results["only_b_cdn"] = (
            self.books_cdn.loc[only_b_cdn]  if only_b_cdn else pd.DataFrame()
        )
        self.results["only_p_cdn"] = (
            self.portal_cdn.loc[only_p_cdn] if only_p_cdn else pd.DataFrame()
        )

        n_confirmed = sum(1 for p in cdn_pairs if p.get("Is_Confirmed"))
        n_review    = len(cdn_pairs) - n_confirmed
        print(f"  CDN pairs — confirmed : {n_confirmed:,}")
        print(f"  CDN pairs — review    : {n_review:,}")
        print(f"  Only in Books  (CDN)  : {len(only_b_cdn):,}")
        print(f"  Only in Portal (CDN)  : {len(only_p_cdn):,}")

        if not self.results["cdn_pairs"].empty:
            for k, v in self.results["cdn_pairs"]["Match_Type"].value_counts().items():
                print(f"    {k:<28}: {v:>6,}")

    # ----------------------------------------------------------
    # Stage 4 -- Coverage Verification (renamed; CDN stream added)
    # ----------------------------------------------------------

    def stage4_coverage(self):
        print("\nSTAGE 4 : Coverage Verification")
        print("-" * 72)

        # ── Invoice stream ─────────────────────────────────────────
        total_p_inv = len(self.portal_inv)
        total_b_inv = len(self.books_inv)
        gstin_p     = self.results["gstin_level_p_count"]
        gstin_b     = self.results["gstin_level_b_count"]

        # gstin counts include CDNs for GSTIN-Level Match suppliers — subtract
        gstin_p_cdn_count = len(self.portal_cdn[
            self.portal_cdn["Supplier GSTIN"].isin(self.gstin_level_match_gstins)])
        gstin_b_cdn_count = len(self.books_cdn[
            self.books_cdn["Supplier GSTIN"].isin(self.gstin_level_match_gstins)])
        gstin_p_inv = gstin_p - gstin_p_cdn_count
        gstin_b_inv = gstin_b - gstin_b_cdn_count

        confirmed = len(self.results.get("matched_b_ids", set()))
        n_op      = len(self.results.get("only_portal", pd.DataFrame()))
        n_ob      = len(self.results.get("only_books",  pd.DataFrame()))

        p_check = gstin_p_inv + confirmed + n_op
        b_check = gstin_b_inv + confirmed + n_ob
        p_ok    = p_check == total_p_inv
        b_ok    = b_check == total_b_inv

        print(f"  Invoice stream:")
        print(f"    Portal: GSTIN-Inv({gstin_p_inv:,}) + Matched({confirmed:,}) + "
              f"OnlyPortal({n_op:,}) = {p_check:,} | Input: {total_p_inv:,} "
              f"{'OK' if p_ok else 'MISMATCH'}")
        print(f"    Books : GSTIN-Inv({gstin_b_inv:,}) + Matched({confirmed:,}) + "
              f"OnlyBooks({n_ob:,}) = {b_check:,} | Input: {total_b_inv:,} "
              f"{'OK' if b_ok else 'MISMATCH'}")

        # ── CDN stream ─────────────────────────────────────────────
        total_p_cdn   = len(self.portal_cdn)
        total_b_cdn   = len(self.books_cdn)
        cdn_df        = self.results.get("cdn_pairs",  pd.DataFrame())
        only_b_cdn_df = self.results.get("only_b_cdn", pd.DataFrame())
        only_p_cdn_df = self.results.get("only_p_cdn", pd.DataFrame())
        n_cdn_matched = len(cdn_df)

        p_cdn_check = n_cdn_matched + len(only_p_cdn_df) + gstin_p_cdn_count
        b_cdn_check = n_cdn_matched + len(only_b_cdn_df) + gstin_b_cdn_count
        p_cdn_ok    = p_cdn_check == total_p_cdn
        b_cdn_ok    = b_cdn_check == total_b_cdn

        print(f"\n  CDN/DN stream:")
        print(f"    Portal: GSTIN-CDN({gstin_p_cdn_count:,}) + Matched({n_cdn_matched:,}) + "
              f"OnlyPortal({len(only_p_cdn_df):,}) = {p_cdn_check:,} | "
              f"Input: {total_p_cdn:,} {'OK' if p_cdn_ok else 'MISMATCH'}")
        print(f"    Books : GSTIN-CDN({gstin_b_cdn_count:,}) + Matched({n_cdn_matched:,}) + "
              f"OnlyBooks({len(only_b_cdn_df):,}) = {b_cdn_check:,} | "
              f"Input: {total_b_cdn:,} {'OK' if b_cdn_ok else 'MISMATCH'}")

        self.results["verification"] = {
            "total_portal":    total_p_inv,
            "total_books":     total_b_inv,
            "gstin_level_p":   gstin_p_inv,
            "gstin_level_b":   gstin_b_inv,
            "matched":         confirmed,
            "only_portal":     n_op,
            "only_books":      n_ob,
            "portal_ok":       p_ok,
            "books_ok":        b_ok,
            "cdn_matched":     n_cdn_matched,
            "only_p_cdn":      len(only_p_cdn_df),
            "only_b_cdn":      len(only_b_cdn_df),
            "cdn_portal_ok":   p_cdn_ok,
            "cdn_books_ok":    b_cdn_ok,
        }

        all_ok = p_ok and b_ok and p_cdn_ok and b_cdn_ok
        if not all_ok:
            raise ValueError("Coverage verification FAILED — investigate before using results.")
        print("\n  100% COVERAGE CONFIRMED — invoice and CDN streams both verified.")

    def run(self):
        start = datetime.now()
        if not self.load_and_clean():
            print("Aborted -- fix file errors above.")
            return False
        self.stage1_gstin()
        self.stage2_invoice_matching()
        self.stage3_cdn_matching()
        self.stage4_coverage()
        elapsed = (datetime.now() - start).total_seconds()
        print("\n" + "=" * 72)
        print(f"  RECONCILIATION COMPLETE -- {elapsed:.1f}s ({elapsed/60:.1f} min)")
        print("=" * 72)
        return True


# ==============================================================
# SECTION 5 -- REPORT WRITER
# ==============================================================

class GSTReportWriterV91:

    def __init__(self, reconciler):
        self.r   = reconciler
        self.cfg = reconciler.cfg
        self.v   = reconciler.results["verification"]

        # Invoice stream results
        self.mp = reconciler.results.get("matched_pairs", pd.DataFrame())
        self.op = reconciler.results.get("only_portal",   pd.DataFrame())
        self.ob = reconciler.results.get("only_books",    pd.DataFrame())

        # CDN stream results (v9.1)
        self.cdn_mp = reconciler.results.get("cdn_pairs",  pd.DataFrame())
        self.op_cdn = reconciler.results.get("only_p_cdn", pd.DataFrame())
        self.ob_cdn = reconciler.results.get("only_b_cdn", pd.DataFrame())

        # Combined views — used by all exception-facing sheets
        # _drop_internal_cols applied before combining so ob_all/op_all
        # never carry _inv_norm, _inv_compact etc. into any downstream sheet
        self.mp_all = self._combine([self.mp,  self.cdn_mp])
        self.op_all = self._combine([
            self._drop_internal_cols(self.op),
            self._drop_internal_cols(self.op_cdn),
        ])
        self.ob_all = self._combine([
            self._drop_internal_cols(self.ob),
            self._drop_internal_cols(self.ob_cdn),
        ])

    @staticmethod
    def _combine(frames):
        valid = [f for f in frames if f is not None and not f.empty]
        if not valid:
            return pd.DataFrame()
        return pd.concat(valid, ignore_index=True)

    @staticmethod
    def _drop_internal_cols(df):
        if df is None or df.empty:
            return pd.DataFrame()
        return df[[c for c in df.columns if not str(c).startswith("_")]].copy()

    @staticmethod
    def _sum(df, col):
        return round(df[col].sum(), 2) if (df is not None and not df.empty
                                           and col in df.columns) else 0

    def _dashboard_df(self):
        v = self.v
        total_p = v["total_portal"] + len(self.r.portal_cdn)
        total_b = v["total_books"]  + len(self.r.books_cdn)
        matched_all = v["matched"] + v.get("cdn_matched", 0)
        match_rate = (
            ((v["gstin_level_p"] + v["matched"]) / v["total_portal"] * 100)
            if v["total_portal"] else 0
        )
        rows = [
            ["Client",                             self.cfg["client_name"]],
            ["Client GSTIN",                       self.cfg.get("client_gstin", "")],
            ["Return Period",                      self.cfg.get("return_period", "")],
            ["Generated At",                       datetime.now().strftime("%d-%b-%Y %H:%M:%S")],
            ["Engine Version",                     VERSION],
            ["--- INVOICE STREAM ---",             ""],
            ["Portal Invoices",                    v["total_portal"]],
            ["Books Invoices",                     v["total_books"]],
            ["GSTIN-Level Matched Portal",         v["gstin_level_p"]],
            ["GSTIN-Level Matched Books",          v["gstin_level_b"]],
            ["Confirmed Invoice Matches",          v["matched"]],
            ["Only in Portal (Invoices)",          v["only_portal"]],
            ["Only in Books  (Invoices)",          v["only_books"]],
            ["Invoice Match Rate %",               round(match_rate, 2)],
            ["--- CDN/DN STREAM ---",              ""],
            ["Portal CDNs / DNs",                  len(self.r.portal_cdn)],
            ["Books CDNs / DNs",                   len(self.r.books_cdn)],
            ["CDN Pairs Matched",                  v.get("cdn_matched",  0)],
            ["Only in Portal (CDN/DN)",            v.get("only_p_cdn",   0)],
            ["Only in Books  (CDN/DN)",            v.get("only_b_cdn",   0)],
            ["--- TAX TOTALS ---",                 ""],
            ["Books Total Tax (all docs)",         self._sum(self.r.books_df,  "Total Tax")],
            ["Portal Total Tax (all docs)",        self._sum(self.r.portal_df, "Total Tax")],
            ["Only in Books Tax / ITC at Risk",    self._sum(self.ob_all, "Total Tax")],
            ["Only in Portal Tax",                 self._sum(self.op_all, "Total Tax")],
            ["Portal Coverage Verified",           v["portal_ok"]],
            ["Books Coverage Verified",            v["books_ok"]],
            ["CDN Portal Coverage Verified",       v.get("cdn_portal_ok", "")],
            ["CDN Books Coverage Verified",        v.get("cdn_books_ok",  "")],
            ["Design Note",
             "GSTIN Match Report uses all documents (audit). "
             "Invoice matching (Stage 2) and CDN matching (Stage 3) use separate streams. "
             "GSTIN-Level Match suppliers skip both Stage 2 and Stage 3."],
        ]
        if not self.mp_all.empty:
            for k, v2 in self.mp_all["Match_Type"].value_counts().items():
                rows.append([f"Match Type - {k}", int(v2)])
        return pd.DataFrame(rows, columns=["Metric", "Value"])

    def _partywise_df(self):
        frames = []
        if not self.mp_all.empty:
            m = self.mp_all.copy()
            m["Reco Status"] = np.where(m["Is_Confirmed"], "Matched", "Review Required")
            frames.append(m[[
                "Supplier GSTIN", "Supplier Name", "Unit", "Reco Status",
                "Books Invoice", "Books Date", "Books Taxable", "Books Tax",
                "Portal Invoice", "Portal Date", "Portal Taxable", "Portal Tax",
                "Taxable Diff", "Tax Diff", "Match_Type", "Match_Reason",
                "Confidence", "Is_Confirmed"
            ]])
        for df_exc, status, inv_col, date_col, tax_col, taxable_col in [
            (self.ob_all, "Only in Books",  "Invoice No", "Invoice Date", "Total Tax", "Taxable"),
            (self.op_all, "Only in Portal", "Invoice No", "Invoice Date", "Total Tax", "Taxable"),
        ]:
            if df_exc is not None and not df_exc.empty:
                ex = df_exc.copy()
                ex["Reco Status"] = status
                is_books = status == "Only in Books"
                ex["Books Invoice"]  = ex[inv_col]  if is_books else ""
                ex["Books Date"]     = ex[date_col]  if is_books else ""
                ex["Books Taxable"]  = ex[taxable_col] if is_books else ""
                ex["Books Tax"]      = ex[tax_col]   if is_books else ""
                ex["Portal Invoice"] = "" if is_books else ex[inv_col]
                ex["Portal Date"]    = "" if is_books else ex[date_col]
                ex["Portal Taxable"] = "" if is_books else ex[taxable_col]
                ex["Portal Tax"]     = "" if is_books else ex[tax_col]
                for c in ["Taxable Diff", "Tax Diff", "Match_Type",
                           "Match_Reason", "Confidence", "Is_Confirmed"]:
                    ex[c] = ""
                frames.append(ex[[
                    "Supplier GSTIN", "Supplier Name", "Unit", "Reco Status",
                    "Books Invoice", "Books Date", "Books Taxable", "Books Tax",
                    "Portal Invoice", "Portal Date", "Portal Taxable", "Portal Tax",
                    "Taxable Diff", "Tax Diff", "Match_Type", "Match_Reason",
                    "Confidence", "Is_Confirmed"
                ]])
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _query_books_df(self):
        if self.ob_all is None or self.ob_all.empty:
            return pd.DataFrame(columns=["Q.No", "Client Response / Action Taken"])
        cols = ["Doc_Class", "Unit", "Supplier Name", "Supplier GSTIN",
                "Invoice No", "Invoice Date", "Taxable",
                "IGST", "CGST", "SGST", "Total Tax",
                "ITC_IGST", "ITC_CGST", "ITC_SGST", "Action"]
        out = self.ob_all[[c for c in cols if c in self.ob_all.columns]].copy()
        out.insert(0, "Q.No", [f"B-Q{i}" for i in range(1, len(out) + 1)])
        out["Client Response / Action Taken"] = ""
        return out

    def _query_portal_df(self):
        if self.op_all is None or self.op_all.empty:
            return pd.DataFrame(columns=["Q.No", "Client Response / Action Taken"])
        cols = ["Doc_Class", "Supplier Name", "Supplier GSTIN",
                "Invoice No", "Invoice Date", "Taxable",
                "IGST", "CGST", "SGST", "Total Tax", "Doc Type", "Action"]
        out = self.op_all[[c for c in cols if c in self.op_all.columns]].copy()
        out.insert(0, "Q.No", [f"P-Q{i}" for i in range(1, len(out) + 1)])
        out["Client Response / Action Taken"] = ""
        return out

    def _fuzzy_review_df(self):
        if self.mp_all.empty:
            return pd.DataFrame(columns=["No entries requiring review"])
        mt   = self.mp_all["Match_Type"].astype(str)
        conf = self.mp_all["Is_Confirmed"].astype(bool)

        # Invoice fuzzy/fragment/ambiguous: include regardless of confirmation
        # — reviewer should see all near-matches for invoice stream
        inv_fuzzy_mask = mt.str.contains(
            "Fuzzy|Fragment|Ambiguous", case=False, na=False
        )
        # CDN matches: include ONLY unconfirmed
        # Confirmed CDN_Amount_Match = engine is confident, no human action needed
        # CDN_Ambiguous always included regardless of confirmation flag
        cdn_needs_review = (
            mt.str.contains("CDN_Ambiguous", case=False, na=False) |
            (mt.str.contains("CDN_Amount_Match", case=False, na=False) & ~conf)
        )
        mask = inv_fuzzy_mask | cdn_needs_review
        out  = self.mp_all[mask].copy()

        if out.empty:
            return pd.DataFrame(columns=["No entries requiring review"])
        out["Review Band"] = np.select(
            [~out["Is_Confirmed"],
             out["Confidence"] >= 90,
             out["Confidence"] >= 80],
            ["REVIEW REQUIRED", "HIGH CONFIDENCE", "MEDIUM CONFIDENCE"],
            default="LOW CONFIDENCE"
        )
        cols = [
            "Review Band", "Is_Confirmed", "Supplier GSTIN", "Supplier Name",
            "Books Invoice", "Books Date", "Books Taxable", "Books Tax",
            "Portal Invoice", "Portal Date", "Portal Taxable", "Portal Tax",
            "Amt Diff", "Tax Abs Diff", "Date Diff Days",
            "Similarity", "Confidence", "Match_Type", "Match_Reason",
            "Books_TXN_ID", "Portal_TXN_ID"
        ]
        return (out[[c for c in cols if c in out.columns]]
                .sort_values(["Is_Confirmed", "Confidence"],
                             ascending=[True, False]))

    def _monthly_itc_analysis_df(self):
        # Monthly analysis uses full dataframes (all documents)
        b = self.r.books_df.copy()
        p = self.r.portal_df.copy()
        b["Month"] = b["Invoice Date"].dt.to_period("M").astype(str)
        p["Month"] = p["Invoice Date"].dt.to_period("M").astype(str)
        b_agg = b.groupby("Month").agg(
            Books_Count=("Invoice No",  "count"),
            Books_Taxable=("Taxable",   "sum"),
            Books_Tax=("Total Tax",     "sum")
        ).reset_index()
        p_agg = p.groupby("Month").agg(
            Portal_Count=("Invoice No", "count"),
            Portal_Taxable=("Taxable",  "sum"),
            Portal_Tax=("Total Tax",    "sum")
        ).reset_index()
        out = p_agg.merge(b_agg, on="Month", how="outer").fillna(0)
        out["Diff_Taxable_Books_minus_Portal"] = (
            out["Books_Taxable"] - out["Portal_Taxable"]).round(2)
        out["Diff_Tax_Books_minus_Portal"] = (
            out["Books_Tax"] - out["Portal_Tax"]).round(2)
        return out.sort_values("Month")

    def _itc_ageing_df(self):
        if self.ob_all is None or self.ob_all.empty:
            return pd.DataFrame(columns=["No only-in-books entries"])
        ob    = self.ob_all.copy()
        today = pd.Timestamp(datetime.now().date())
        ob["Age Days"] = (today - ob["Invoice Date"]).dt.days
        ob["Age Bucket"] = pd.cut(
            ob["Age Days"],
            bins=[-999999, 30, 60, 90, 180, 999999],
            labels=["0-30", "31-60", "61-90", "91-180", ">180"]
        )

        # Split: positive entries = ITC at risk (invoice not on portal)
        #        negative entries = CDN/DN not matched (reversal may be needed)
        # Both reported but separated so ITC risk picture is not distorted
        pos = ob[ob["Total Tax"] > 0].copy()
        neg = ob[ob["Total Tax"] < 0].copy()

        separator = pd.DataFrame([{
            "Age Bucket": "---",
            "Age Days":   "---",
            "Doc_Class":  "--- CREDIT / DEBIT NOTES UNMATCHED (reversal may be required) ---",
            "Supplier Name": "", "Supplier GSTIN": "",
            "Invoice No": "", "Invoice Date": "",
            "Taxable": "", "IGST": "", "CGST": "", "SGST": "",
            "Total Tax": "", "Action": "",
        }])

        cols = ["Age Bucket", "Age Days", "Doc_Class", "Supplier Name",
                "Supplier GSTIN", "Invoice No", "Invoice Date",
                "Taxable", "IGST", "CGST", "SGST", "Total Tax", "Action"]

        pos_out = pos[[c for c in cols if c in pos.columns]].sort_values(
            ["Age Days", "Total Tax"], ascending=[False, False])
        neg_out = neg[[c for c in cols if c in neg.columns]].sort_values(
            ["Age Days", "Total Tax"], ascending=[False, True])

        if neg_out.empty:
            return pos_out
        return pd.concat(
            [pos_out, separator[[c for c in cols if c in separator.columns]], neg_out],
            ignore_index=True
        )

    def _llm_ready_summary_df(self):
        v = self.v
        rows = [
            {"Section": "Summary", "Key": "Client",                "Value": self.cfg["client_name"]},
            {"Section": "Summary", "Key": "Period",                "Value": self.cfg.get("return_period", "")},
            {"Section": "Summary", "Key": "Only in Books Count",   "Value": v["only_books"]},
            {"Section": "Summary", "Key": "Only in Portal Count",  "Value": v["only_portal"]},
            {"Section": "Summary", "Key": "CDN Only Books Count",  "Value": v.get("only_b_cdn", 0)},
            {"Section": "Summary", "Key": "CDN Only Portal Count", "Value": v.get("only_p_cdn", 0)},
            {"Section": "Summary", "Key": "Confirmed Inv Matches", "Value": v["matched"]},
            {"Section": "Summary", "Key": "CDN Matches",           "Value": v.get("cdn_matched", 0)},
            {"Section": "Summary", "Key": "ITC at Risk (Books)",
             "Value": round(self._sum(self.ob_all, "Total Tax"), 2)},
            {"Section": "Summary", "Key": "ITC Available (Portal)",
             "Value": round(self._sum(self.op_all, "Total Tax"), 2)},
        ]

        # Match type breakup
        if not self.mp_all.empty and "Match_Type" in self.mp_all.columns:
            for k, val in self.mp_all["Match_Type"].value_counts().items():
                rows.append({"Section": "Match Breakup", "Key": k, "Value": int(val)})

        # Top exceptions — guarded for missing columns
        for label, df, issue in [
            ("Top Only in Books",  self.ob_all, "In books but not in GSTR-2B"),
            ("Top Only in Portal", self.op_all, "In GSTR-2B but not in books"),
        ]:
            if df is None or df.empty:
                continue
            if "Total Tax" not in df.columns:
                continue
            try:
                top = df[df["Total Tax"].notna()].sort_values(
                    "Total Tax", ascending=False).head(25)
            except Exception:
                continue
            for _, r in top.iterrows():
                try:
                    rows.append({
                        "Section": label,
                        "Key":     str(r.get("TXN_ID", "")),
                        "Value":   json.dumps({
                            "doc_class": str(r.get("Doc_Class",      "")),
                            "supplier":  str(r.get("Supplier Name",  "")),
                            "gstin":     str(r.get("Supplier GSTIN", "")),
                            "invoice":   str(r.get("Invoice No",     "")),
                            "date":      str(r.get("Invoice Date",   "")),
                            "taxable":   float(r.get("Taxable",   0) or 0),
                            "tax":       float(r.get("Total Tax", 0) or 0),
                            "issue":     issue,
                        }, default=str),
                    })
                except Exception:
                    continue
        return pd.DataFrame(rows)

    def _style_workbook(self, path):
        wb = load_workbook(path)
        thin      = Side(style="thin", color="CCCCCC")
        border    = Border(left=thin, right=thin, top=thin, bottom=thin)
        hdr_fill  = PatternFill("solid", fgColor="1F3864")
        hdr_font  = Font(name="Arial", color="FFFFFF", bold=True, size=9)
        body_font = Font(name="Arial", color="000000", size=9)

        for ws in wb.worksheets:
            ws.sheet_view.showGridLines = False
            ws.freeze_panes = "A2"
            for cell in ws[1]:
                cell.fill      = hdr_fill
                cell.font      = hdr_font
                cell.alignment = Alignment(horizontal="center",
                                           vertical="center", wrap_text=True)
                cell.border    = border
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.font      = body_font
                    cell.alignment = Alignment(vertical="center", wrap_text=False)
                    cell.border    = border
            for col_idx, col_cells in enumerate(ws.columns, start=1):
                max_len = 10
                for cell in col_cells:
                    val     = "" if cell.value is None else str(cell.value)
                    max_len = max(max_len, min(len(val), 60))
                ws.column_dimensions[get_column_letter(col_idx)].width = max_len + 2
        wb.save(path)

    def write(self):
        print("\nGENERATING REPORT")
        print("-" * 72)
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = os.path.join(
            self.cfg["base_path"],
            f"ITC_Reconciliation_v9_1_{ts}.xlsx"
        )

        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            self._dashboard_df().to_excel(
                writer, sheet_name="Dashboard", index=False)
            self.r.results["gstin_summary"].to_excel(
                writer, sheet_name="GSTIN Match Report", index=False)
            self._partywise_df().to_excel(
                writer, sheet_name="Party-wise Reco", index=False)
            self._query_books_df().to_excel(
                writer, sheet_name="Query Sheet -- Books", index=False)
            self._query_portal_df().to_excel(
                writer, sheet_name="Query Sheet -- Portal", index=False)
            self.ob_all.to_excel(
                writer, sheet_name="Only in Books B-A", index=False)
            self.op_all.to_excel(
                writer, sheet_name="Only in Portal A-B", index=False)
            self._fuzzy_review_df().to_excel(
                writer, sheet_name="Fuzzy Match Review", index=False)
            self._monthly_itc_analysis_df().to_excel(
                writer, sheet_name="Monthly ITC Analysis", index=False)
            self._itc_ageing_df().to_excel(
                writer, sheet_name="ITC Ageing", index=False)
            self._llm_ready_summary_df().to_excel(
                writer, sheet_name="LLM Ready Summary", index=False)

        self._style_workbook(path)
        kb = os.path.getsize(path) / 1024
        print("  OK  Dashboard            (CDN metrics added)")
        print("  OK  GSTIN Match Report   (unchanged)")
        print("  OK  Party-wise Reco      (invoices + CDNs combined)")
        print("  OK  Query Sheets         (Doc_Class column added)")
        print("  OK  Only in Books / Portal (invoices + CDNs)")
        print("  OK  Fuzzy Match Review   (CDN types included)")
        print("  OK  Monthly ITC Analysis")
        print("  OK  ITC Ageing           (invoices + CDNs)")
        print("  OK  LLM Ready Summary    (doc_class added to payloads)")
        print(f"\n  Saved : {os.path.basename(path)}")
        print(f"  Path  : {path}")
        print(f"  Size  : {kb:.1f} KB")
        return path


# ==============================================================
# SECTION 6 -- MAIN
# ==============================================================

if __name__ == "__main__":
    print(f"GST ITC RECONCILIATION {VERSION}  --  Y K JONEJA & CO")
    print("=" * 72)

    all_ok = True
    for label, path in [("GSTR-2B", GSTR2B_PATH), ("Purchase", PURCHASE_PATH)]:
        if os.path.exists(path):
            mb = os.path.getsize(path) / (1024 * 1024)
            print(f"  OK  {label:<12} ({mb:.2f} MB)")
        else:
            print(f"  ERR {label:<12} NOT FOUND")
            print(f"      Expected : {path}")
            all_ok = False

    if not all_ok:
        print("\nFix file paths in CONFIG section and retry.")
        sys.exit(1)

    print()
    if not validate_columns():
        print("\nFix column mapping and retry.")
        sys.exit(1)

    cfg = {
        "client_name":    CLIENT_NAME,
        "client_gstin":   CLIENT_GSTIN,
        "return_period":  RETURN_PERIOD,
        "base_path":      BASE_PATH,
        "gstr2b_path":    GSTR2B_PATH,
        "gstr2b_sheet":   GSTR2B_SHEET,
        "purchase_path":  PURCHASE_PATH,
        "purchase_sheet": PURCHASE_SHEET,
        "amount_tolerance":               AMOUNT_TOLERANCE,
        "tax_tolerance":                  TAX_TOLERANCE,
        "date_tolerance":                 DATE_TOLERANCE,
        "similarity_threshold":           SIMILARITY_THRESHOLD,
        "fragment_threshold":             FRAGMENT_THRESHOLD,
        "short_numeric_max_digits":       SHORT_NUMERIC_MAX_DIGITS,
        "short_numeric_date_tolerance":   SHORT_NUMERIC_DATE_TOLERANCE,
        "needs_review_confidence_gap":    NEEDS_REVIEW_CONFIDENCE_GAP,
        "ambiguous_matches_are_unconfirmed": AMBIGUOUS_MATCHES_ARE_UNCONFIRMED,
        # v9.1
        "cdn_tax_tolerance":  CDN_TAX_TOLERANCE,
        "cdn_date_tolerance": CDN_DATE_TOLERANCE,
        "portal_cols":        PORTAL_COLS,
        "books_cols":         BOOKS_COLS,
    }

    recon = GSTReconcilerV91(cfg)
    ok    = recon.run()

    if ok:
        writer      = GSTReportWriterV91(recon)
        report_path = writer.write()
        print("\nReport saved at:")
        print(f"  {report_path}")
    else:
        print("Reconciliation failed -- fix errors above.")
        sys.exit(1)


In [ ]:
# ==============================================================
# ITC RECONCILIATION INTELLIGENCE LAYER
# Y K JONEJA & CO  --  Faridabad
#
# Reads the v9.1 reconciliation Excel output, runs a 4-agent
# CrewAI crew, and writes results back into the same workbook.
#
# Agents:
#   Agent 2  Escape Investigator
#            Reads: "Only in Books B-A" + "Only in Portal A-B"
#            Writes: "Escape Candidates" sheet (new)
#
#   Agent 3  Fuzzy Match Explainer + Risk Ranker
#            Reads: "Fuzzy Match Review"
#            Writes: "Fuzzy Match Triage" sheet (new)
#
#   Agent 4  Compliance Checker
#            Reads: "Escape Candidates" + "Fuzzy Match Triage"
#            Writes: "Compliance Flags" sheet (new)
#
#   Agent 5  Supplier Query Drafter
#            Reads: "Escape Candidates" + "Compliance Flags"
#            Writes: "Supplier Query Draft" sheet (new)
#
# Usage:
#   python itc_recon_crew.py <path_to_v9_1_output.xlsx>
#
# Requirements:
#   pip install crewai==0.80.0 langchain-google-genai openpyxl pandas
#   Set GOOGLE_API_KEY in environment or .env file
#
# ==============================================================

import os
import sys
import json
import re
import textwrap
from datetime import datetime
from pathlib import Path

import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

from crewai import Agent, Task, Crew, Process
from langchain_google_genai import ChatGoogleGenerativeAI


# ==============================================================
# SECTION 1 -- CONFIG
# ==============================================================

# Models
MODEL_REASONING = "gemini-2.5-pro"    # Agents 2 & 3 — multi-signal judgment
MODEL_DRAFTING  = "gemini-2.5-flash"  # Agents 4 & 5 — structured drafting

# Pre-filter tolerances (loose — candidate generation only)
PREFILTER_TAX_TOL  = 500.0    # Rs
PREFILTER_DATE_TOL = 90       # days
PREFILTER_MIN_STR_OVERLAP = 3  # chars (compact form)

# Styling (matches v9.1 workbook)
HDR_DARK  = "1F3864"
HDR_MID   = "2E75B6"
WHITE     = "FFFFFF"
BLUE_LITE = "DEEAF1"
AMBER     = "FCE4D6"
RED       = "FFCCCC"
GREEN     = "E2EFDA"
YELLOW    = "FFFF99"

_thin = Side(style="thin", color="CCCCCC")
_BDR  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)


# ==============================================================
# SECTION 2 -- STYLING HELPERS
# ==============================================================

def _hdr(cell, text, bg=HDR_DARK, fg=WHITE, size=9):
    cell.value     = text
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.font      = Font(name="Arial", color=fg, bold=True, size=size)
    cell.alignment = Alignment(horizontal="center", vertical="center",
                               wrap_text=True)
    cell.border    = _BDR


def _body(cell, value, bg=WHITE, bold=False, color="000000",
          halign="left", fmt=None):
    cell.value     = value
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.font      = Font(name="Arial", color=color, bold=bold, size=9)
    cell.alignment = Alignment(horizontal=halign, vertical="center",
                               wrap_text=False)
    cell.border    = _BDR
    if fmt:
        cell.number_format = fmt


def _col_w(ws, col_idx, width):
    ws.column_dimensions[get_column_letter(col_idx)].width = width


# ==============================================================
# SECTION 3 -- PRE-FILTER  (Python function, not an agent)
# ==============================================================

def _compact(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())


def _str_overlap(a, b):
    """Longest common substring length on compact forms."""
    a, b = _compact(a), _compact(b)
    best = 0
    for i in range(len(a)):
        for j in range(len(b)):
            k = 0
            while (i + k < len(a) and j + k < len(b)
                   and a[i + k] == b[j + k]):
                k += 1
            best = max(best, k)
    return best


def run_prefilter(books_df: pd.DataFrame,
                  portal_df: pd.DataFrame) -> list[dict]:
    """
    Groups by GSTIN.  For each GSTIN compares every books row against
    every portal row using loose tolerances.  Returns candidate pairs
    as a list of dicts suitable for JSON serialisation.
    """
    candidates = []
    pair_id    = 0

    all_gstins = set(books_df["Supplier GSTIN"].dropna().unique()) & \
                 set(portal_df["Supplier GSTIN"].dropna().unique())

    for gstin in all_gstins:
        b_rows = books_df[books_df["Supplier GSTIN"] == gstin]
        p_rows = portal_df[portal_df["Supplier GSTIN"] == gstin]

        for _, b in b_rows.iterrows():
            for _, p in p_rows.iterrows():
                b_tax  = abs(float(b.get("Total Tax", 0) or 0))
                p_tax  = abs(float(p.get("Total Tax", 0) or 0))
                tax_diff = abs(b_tax - p_tax)
                if tax_diff > PREFILTER_TAX_TOL:
                    continue

                b_date = b.get("Invoice Date")
                p_date = p.get("Invoice Date")
                try:
                    date_diff = abs((pd.Timestamp(b_date)
                                     - pd.Timestamp(p_date)).days)
                except Exception:
                    date_diff = 999
                if date_diff > PREFILTER_DATE_TOL:
                    continue

                overlap = _str_overlap(
                    str(b.get("Invoice No", "")),
                    str(p.get("Invoice No", ""))
                )

                # Must pass at least one signal beyond GSTIN + tax
                if tax_diff > 0 and date_diff > 7 and overlap < PREFILTER_MIN_STR_OVERLAP:
                    continue

                signals = []
                if tax_diff == 0:
                    signals.append("tax_exact")
                elif tax_diff <= 50:
                    signals.append(f"tax_diff_{tax_diff:.0f}")
                if date_diff == 0:
                    signals.append("date_exact")
                elif date_diff <= 7:
                    signals.append(f"date_diff_{date_diff}d")
                if overlap >= PREFILTER_MIN_STR_OVERLAP:
                    signals.append(f"str_overlap_{overlap}ch")

                pair_id += 1
                candidates.append({
                    "pair_id":         pair_id,
                    "gstin":           gstin,
                    "supplier_name":   str(b.get("Supplier Name", "")),
                    "books_inv":       str(b.get("Invoice No", "")),
                    "books_date":      str(b.get("Invoice Date", ""))[:10],
                    "books_taxable":   float(b.get("Taxable", 0) or 0),
                    "books_tax":       float(b.get("Total Tax", 0) or 0),
                    "portal_inv":      str(p.get("Invoice No", "")),
                    "portal_date":     str(p.get("Invoice Date", ""))[:10],
                    "portal_taxable":  float(p.get("Taxable", 0) or 0),
                    "portal_tax":      float(p.get("Total Tax", 0) or 0),
                    "tax_diff":        round(tax_diff, 2),
                    "date_diff_days":  date_diff,
                    "prefilter_signal": " | ".join(signals),
                })

    return candidates


# ==============================================================
# SECTION 4 -- LLM INITIALISATION
# ==============================================================

def _llm(model: str):
    api_key = os.environ.get("GOOGLE_API_KEY", "")
    if not api_key:
        raise EnvironmentError(
            "GOOGLE_API_KEY not set. "
            "Export it or add to .env before running."
        )
    return ChatGoogleGenerativeAI(
        model=model,
        google_api_key=api_key,
        temperature=0.2,       # low temp — factual, consistent
        convert_system_message_to_human=True,
    )


# ==============================================================
# SECTION 5 -- AGENT & TASK DEFINITIONS
# ==============================================================

def build_crew(candidates_json: str,
               fuzzy_json: str,
               client_name: str,
               return_period: str) -> Crew:
    """
    Builds and returns the CrewAI crew.
    All context is passed as strings to avoid tool dependencies.
    """

    llm_pro   = _llm(MODEL_REASONING)
    llm_flash = _llm(MODEL_DRAFTING)

    # ── Agent 2: Escape Investigator ──────────────────────────
    agent2 = Agent(
        role="GST ITC Escape Investigator",
        goal=(
            "Examine candidate pairs where invoices appear in books "
            "but not in GSTR-2B (or vice versa) and determine whether "
            "each pair is a genuine match that escaped automated "
            "reconciliation, a possible match requiring human review, "
            "or a coincidental pairing that should stay as an exception."
        ),
        backstory=(
            "You are a senior GST reconciliation analyst at a tax "
            "practice in Faridabad with 15 years of experience. "
            "You understand that invoice numbering varies widely "
            "across ERPs — Tally uses slash separators, suppliers "
            "sometimes upload only the serial component to the GST "
            "portal, financial year segments appear in different formats "
            "(24-25, 2425, 2024-25), and credit note prefixes (CR, CDN, "
            "CN) get reordered. Your job is to look past these cosmetic "
            "differences and identify the economic substance: are these "
            "the same underlying transaction?"
        ),
        llm=llm_pro,
        verbose=True,
        allow_delegation=False,
    )

    task2 = Task(
        description=textwrap.dedent(f"""
            Client: {client_name}  |  Period: {return_period}

            Below are candidate pairs identified by the pre-filter.
            Each pair has one row from "Only in Books" and one from
            "Only in Portal" for the same supplier GSTIN.

            CANDIDATE PAIRS (JSON):
            {candidates_json}

            For EACH pair produce a JSON object with these keys:
              pair_id       : integer (from input)
              verdict       : one of "GENUINE_MATCH" | "POSSIBLE_MATCH" | "COINCIDENTAL"
              explanation   : 2-3 sentences in plain English explaining your
                              reasoning — what signals matched, what differs,
                              what the accounts team should do.
              action        : one of "Accept match — update books" |
                              "Verify with supplier" |
                              "Keep as exception"

            Return ONLY a JSON array of these objects.
            No preamble, no markdown fences.
        """),
        agent=agent2,
        expected_output=(
            "JSON array of verdict objects, one per pair_id. "
            "Keys: pair_id, verdict, explanation, action."
        ),
    )

    # ── Agent 3: Fuzzy Match Explainer + Risk Ranker ───────────
    agent3 = Agent(
        role="GST Fuzzy Match Analyst",
        goal=(
            "Translate cryptic engine match-reason strings into plain "
            "English explanations a CA or advocate can read in 10 seconds, "
            "separate CDN rows from invoice rows, and rank the entire "
            "review queue by financial risk so the practitioner reviews "
            "the most important rows first."
        ),
        backstory=(
            "You work in a GST practice and routinely review fuzzy "
            "reconciliation outputs. You know that 'Compact invoice "
            "substring match' means the invoice numbers are identical "
            "once separators are stripped. You know 'CDN_Ambiguous' "
            "means two credit notes have similar amounts and the engine "
            "cannot distinguish them — the supplier's accounts team must "
            "confirm. You rank by: unconfirmed high-tax rows first, "
            "then CDN_Ambiguous, then Needs_Review confirmed."
        ),
        llm=llm_pro,
        verbose=True,
        allow_delegation=False,
    )

    task3 = Task(
        description=textwrap.dedent(f"""
            Client: {client_name}  |  Period: {return_period}

            Below are rows from the Fuzzy Match Review sheet.

            FUZZY MATCH ROWS (JSON):
            {fuzzy_json}

            For EACH row produce a JSON object with:
              row_id        : integer (use order in input, 1-based)
              risk_rank     : integer (1 = highest risk)
              risk_level    : "HIGH" | "MEDIUM" | "LOW"
              doc_class     : "Invoice" | "Credit Note" | "Debit Note"
              explanation   : 2-3 sentences plain English — what the
                              engine found, why it needs review, what
                              the reviewer should check.

            Risk ranking rules:
              - Unconfirmed rows with Total Tax > 50000 → HIGH
              - CDN_Ambiguous regardless of amount → HIGH
              - Needs_Review unconfirmed, tax 10000–50000 → MEDIUM
              - Confirmed rows or tax < 10000 → LOW
              - Within same risk level, rank by Total Tax descending.

            Return ONLY a JSON array. No preamble, no markdown fences.
        """),
        agent=agent3,
        expected_output=(
            "JSON array of risk-ranked objects. "
            "Keys: row_id, risk_rank, risk_level, doc_class, explanation."
        ),
    )

    # ── Agent 4: Compliance Checker ────────────────────────────
    agent4 = Agent(
        role="GST Compliance Analyst",
        goal=(
            "Review ITC exceptions and fuzzy match issues, identify "
            "the applicable GST provisions, assess the legal risk to "
            "the taxpayer, and recommend a specific compliance action."
        ),
        backstory=(
            "You are an advocate and GST consultant with deep knowledge "
            "of the CGST Act 2017 and CGST Rules 2017. You know that "
            "Section 16(2)(c) blocks ITC when a supplier has not filed "
            "their GSTR-1/GSTR-3B. You know Rule 37 requires ITC reversal "
            "on unpaid invoices beyond 180 days. You know Section 34 "
            "governs credit notes and their time limits. You know Rule 46 "
            "mandates specific particulars on a tax invoice. You flag "
            "issues with the precise provision reference — not generic "
            "advice — because your client's advocate will use this to "
            "draft responses to notices."
        ),
        llm=llm_flash,
        verbose=True,
        allow_delegation=False,
    )

    task4 = Task(
        description=textwrap.dedent(f"""
            Client: {client_name}  |  Period: {return_period}

            You are given two inputs:

            1. ESCAPE VERDICTS from Agent 2 (pairs that escaped matching):
            {candidates_json}

            2. FUZZY MATCH RISK RANKS from Agent 3:
            {fuzzy_json}

            For each GENUINE_MATCH or POSSIBLE_MATCH from Agent 2, and
            for each HIGH or MEDIUM risk row from Agent 3, produce a
            compliance flag JSON object with:
              flag_id           : "CF-XX" sequential
              supplier_name     : string
              document_ref      : invoice or CN number
              issue_category    : brief category label
              itc_at_risk_rs    : float (0 if no direct ITC risk)
              provision         : exact section/rule (e.g. "Section 16(2)(c) CGST Act 2017")
              rule_ref          : short ref (e.g. "Sec 16(2)(c)")
              risk_level        : "CRITICAL" | "HIGH" | "MEDIUM" | "LOW"
              time_sensitivity  : when action is needed
              recommended_action: specific action in 1-2 sentences
              compliance_note   : 2-3 sentences explaining the legal risk

            Use ONLY provisions from CGST Act 2017 and CGST Rules 2017.
            Do NOT cite case law — provisions only at this stage.
            Return ONLY a JSON array. No preamble, no markdown fences.
        """),
        agent=agent4,
        expected_output=(
            "JSON array of compliance flag objects with exact provision "
            "citations from CGST Act / Rules."
        ),
        context=[task2, task3],
    )

    # ── Agent 5: Supplier Query Drafter ────────────────────────
    agent5 = Agent(
        role="GST Query Letter Drafter",
        goal=(
            "Draft clear, professional, supplier-wise query paragraphs "
            "for the client's accounts team to send to their suppliers. "
            "Each query must state the document reference, the discrepancy, "
            "and the specific information or action required."
        ),
        backstory=(
            "You draft GST query letters for a tax practice. The letters "
            "go to the client's accounts team (not directly to suppliers) "
            "so the tone is clear and instructional. Each query paragraph "
            "must be self-contained — the accounts person should be able "
            "to copy it into an email to the supplier without needing "
            "any additional context. Never use legal jargon in query "
            "letters — use plain business language. Always state the "
            "invoice number, date, amount, and what specific response "
            "is required."
        ),
        llm=llm_flash,
        verbose=True,
        allow_delegation=False,
    )

    task5 = Task(
        description=textwrap.dedent(f"""
            Client: {client_name}  |  Period: {return_period}

            Draft supplier-wise query paragraphs based on the compliance
            flags from Agent 4.

            Group all flags by supplier_name.
            For each supplier produce a JSON object with:
              supplier_name   : string
              supplier_gstin  : string (from flag data)
              query_paras     : list of objects, one per flag, each with:
                  flag_id         : from compliance flag
                  document_ref    : invoice / CN number
                  query_para      : 3-5 sentences. Plain English.
                                    State: what document, what discrepancy,
                                    what information is needed, by when.
                                    Do NOT use section numbers or legal terms.

            Also produce one supplier_summary per supplier:
              supplier_summary: 1-2 sentences summarising total ITC at
                                risk and number of open items for this
                                supplier.

            Return ONLY a JSON array (one object per supplier).
            No preamble, no markdown fences.
        """),
        agent=agent5,
        expected_output=(
            "JSON array, one object per supplier, with query_paras list "
            "and supplier_summary."
        ),
        context=[task4],
    )

    crew = Crew(
        agents=[agent2, agent3, agent4, agent5],
        tasks=[task2, task3, task4, task5],
        process=Process.sequential,
        verbose=2,
    )

    return crew


# ==============================================================
# SECTION 6 -- EXCEL READER
# ==============================================================

def read_sheets(xlsx_path: str) -> dict:
    """Read relevant sheets from the v9.1 output workbook."""
    required = {
        "Only in Books B-A":  "books_exceptions",
        "Only in Portal A-B": "portal_exceptions",
        "Fuzzy Match Review":  "fuzzy_review",
        "LLM Ready Summary":   "llm_summary",
    }
    sheets = {}
    xl = pd.ExcelFile(xlsx_path)
    available = xl.sheet_names

    for sheet_name, key in required.items():
        if sheet_name in available:
            df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
            # Parse dates
            for col in df.columns:
                if "date" in col.lower() or "Date" in col:
                    df[col] = pd.to_datetime(df[col], errors="coerce")
            sheets[key] = df
            print(f"  READ  {sheet_name:<30} {len(df):>5} rows")
        else:
            print(f"  WARN  {sheet_name} not found — skipping")
            sheets[key] = pd.DataFrame()

    # Extract client metadata from LLM Ready Summary
    meta = {"client_name": "Client", "return_period": ""}
    if not sheets["llm_summary"].empty:
        summary_df = sheets["llm_summary"]
        for _, row in summary_df.iterrows():
            if str(row.get("Key", "")) == "Client":
                meta["client_name"] = str(row.get("Value", "Client"))
            if str(row.get("Key", "")) == "Period":
                meta["return_period"] = str(row.get("Value", ""))

    sheets["meta"] = meta
    return sheets


# ==============================================================
# SECTION 7 -- JSON SERIALISERS FOR AGENT CONTEXT
# ==============================================================

def _df_to_json(df: pd.DataFrame, max_rows: int = 200) -> str:
    """Serialise a dataframe to a compact JSON string for agent context."""
    if df is None or df.empty:
        return "[]"
    # Drop internal columns (prefixed with _)
    cols = [c for c in df.columns if not str(c).startswith("_")]
    subset = df[cols].head(max_rows).copy()
    # Convert dates to strings
    for col in subset.select_dtypes(include=["datetime64[ns]",
                                              "datetime64[ns, UTC]"]).columns:
        subset[col] = subset[col].dt.strftime("%Y-%m-%d")
    subset = subset.fillna("")
    return subset.to_json(orient="records", indent=None)


def candidates_to_json(candidates: list[dict]) -> str:
    return json.dumps(candidates, indent=None, default=str)


# ==============================================================
# SECTION 8 -- EXCEL WRITER (writes agent output back to workbook)
# ==============================================================

def _parse_json_response(raw: str) -> list:
    """Safely parse JSON from agent output — strips markdown fences."""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Try to extract first [...] block
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except Exception:
                pass
    print("  WARN  Could not parse agent JSON — check crew output above")
    return []


def write_escape_candidates_sheet(wb, candidates: list[dict],
                                  verdicts: list[dict]):
    """Write Escape Candidates sheet with Agent 2 verdicts."""
    # Build verdict lookup
    vdict = {v["pair_id"]: v for v in verdicts}

    if "Escape Candidates" in wb.sheetnames:
        del wb["Escape Candidates"]
    ws = wb.create_sheet("Escape Candidates")
    ws.freeze_panes = "A3"
    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:P1")
    _hdr(ws["A1"], "ESCAPE CANDIDATES  |  Agent 2: Escape Investigator",
         size=10)
    ws.row_dimensions[1].height = 22

    headers = [
        "Pair#", "Supplier GSTIN", "Supplier Name",
        "Books Invoice", "Books Date", "Books Taxable (₹)", "Books Tax (₹)",
        "Portal Invoice", "Portal Date", "Portal Taxable (₹)", "Portal Tax (₹)",
        "Tax Diff (₹)", "Date Diff (days)", "Pre-filter Signal",
        "Agent Verdict", "Agent Explanation"
    ]
    for c, h in enumerate(headers, 1):
        _hdr(ws.cell(2, c), h, bg=HDR_MID)
    ws.row_dimensions[2].height = 28

    verdict_colors = {
        "GENUINE_MATCH":  GREEN,
        "POSSIBLE_MATCH": YELLOW,
        "COINCIDENTAL":   AMBER,
    }
    row_bgs = [BLUE_LITE, WHITE]

    for i, pair in enumerate(candidates):
        r   = i + 3
        bg  = row_bgs[i % 2]
        v   = vdict.get(pair["pair_id"], {})
        verdict     = v.get("verdict", "")
        explanation = v.get("explanation", "")
        vbg = verdict_colors.get(verdict, YELLOW)

        vals = [
            pair["pair_id"], pair["gstin"], pair["supplier_name"],
            pair["books_inv"], pair["books_date"],
            pair["books_taxable"], pair["books_tax"],
            pair["portal_inv"], pair["portal_date"],
            pair["portal_taxable"], pair["portal_tax"],
            pair["tax_diff"], pair["date_diff_days"],
            pair["prefilter_signal"],
            verdict, explanation,
        ]
        for c, v_val in enumerate(vals, 1):
            fmt    = "#,##0.00" if c in (6, 7, 10, 11, 12) else None
            halign = "right"    if c in (6, 7, 10, 11, 12) else (
                     "center"   if c in (1, 13) else "left")
            cell_bg = vbg if c in (15, 16) else bg
            _body(ws.cell(r, c), v_val, bg=cell_bg,
                  halign=halign, fmt=fmt)
        ws.row_dimensions[r].height = 18

    widths = [5, 20, 24, 20, 12, 16, 14, 18, 12, 16, 14, 12, 13, 38, 16, 55]
    for c, w in enumerate(widths, 1):
        _col_w(ws, c, w)


def write_fuzzy_triage_sheet(wb, fuzzy_df: pd.DataFrame,
                              triage: list[dict]):
    """Write Fuzzy Match Triage sheet with Agent 3 results."""
    tdict = {t["row_id"]: t for t in triage}

    if "Fuzzy Match Triage" in wb.sheetnames:
        del wb["Fuzzy Match Triage"]
    ws = wb.create_sheet("Fuzzy Match Triage")
    ws.freeze_panes = "A3"
    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:R1")
    _hdr(ws["A1"],
         "FUZZY MATCH TRIAGE  |  Agent 3: Explainer + Risk Ranker",
         size=10)
    ws.row_dimensions[1].height = 22

    headers = [
        "Risk Rank", "Risk Level", "Doc Class",
        "Supplier GSTIN", "Supplier Name",
        "Match Type", "Is Confirmed",
        "Books Invoice", "Books Date", "Books Tax (₹)",
        "Portal Invoice", "Portal Date", "Portal Tax (₹)",
        "Tax Diff (₹)", "Date Diff (days)", "Similarity %", "Confidence",
        "Agent Plain English Explanation"
    ]
    for c, h in enumerate(headers, 1):
        _hdr(ws.cell(2, c), h, bg=HDR_MID)
    ws.row_dimensions[2].height = 28

    risk_bg = {"HIGH": RED, "MEDIUM": AMBER, "LOW": GREEN}

    # Sort fuzzy_df rows by agent risk rank if available
    rows_list = list(fuzzy_df.iterrows())

    for i, (_, row) in enumerate(rows_list):
        r     = i + 3
        t     = tdict.get(i + 1, {})
        rlvl  = t.get("risk_level", "")
        bg    = risk_bg.get(rlvl, WHITE)
        expl  = t.get("explanation", "")
        rank  = t.get("risk_rank", "")
        dcls  = t.get("doc_class", "")

        # Read original columns defensively
        def _g(col):
            return row.get(col, "") if col in row.index else ""

        vals = [
            rank, rlvl, dcls,
            _g("Supplier GSTIN"), _g("Supplier Name"),
            _g("Match_Type"),     _g("Is_Confirmed"),
            _g("Books Invoice"),  _g("Books Date"),  _g("Books Tax"),
            _g("Portal Invoice"), _g("Portal Date"), _g("Portal Tax"),
            _g("Tax Abs Diff"),   _g("Date Diff Days"),
            _g("Similarity"),     _g("Confidence"),
            expl,
        ]
        for c, v_val in enumerate(vals, 1):
            fmt    = "#,##0.00" if c in (10, 13, 14) else None
            halign = "right"    if c in (10, 13, 14) else (
                     "center"   if c in (1, 7, 15, 16, 17) else "left")
            cell_bg = YELLOW if c == 18 else bg
            _body(ws.cell(r, c), v_val, bg=cell_bg,
                  halign=halign, fmt=fmt)
        ws.row_dimensions[r].height = 18

    widths = [8, 9, 12, 20, 24, 18, 12, 20, 12, 14, 18, 12, 14,
              12, 13, 11, 11, 58]
    for c, w in enumerate(widths, 1):
        _col_w(ws, c, w)


def write_compliance_sheet(wb, flags: list[dict]):
    """Write Compliance Flags sheet with Agent 4 results."""
    if "Compliance Flags" in wb.sheetnames:
        del wb["Compliance Flags"]
    ws = wb.create_sheet("Compliance Flags")
    ws.freeze_panes = "A3"
    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:K1")
    _hdr(ws["A1"],
         "COMPLIANCE FLAGS  |  Agent 4: GST Compliance Checker",
         size=10)
    ws.row_dimensions[1].height = 22

    headers = [
        "Flag#", "Supplier Name", "Document Ref",
        "Issue Category", "ITC at Risk (₹)",
        "Applicable Provision", "Rule Ref",
        "Risk Level", "Time Sensitivity",
        "Recommended Action", "Compliance Note"
    ]
    for c, h in enumerate(headers, 1):
        _hdr(ws.cell(2, c), h, bg=HDR_MID)
    ws.row_dimensions[2].height = 28

    risk_bg = {"CRITICAL": RED, "HIGH": AMBER,
               "MEDIUM": YELLOW, "LOW": GREEN}

    for i, flag in enumerate(flags):
        r   = i + 3
        rlvl = flag.get("risk_level", "")
        bg  = risk_bg.get(rlvl, WHITE)

        vals = [
            flag.get("flag_id", ""),
            flag.get("supplier_name", ""),
            flag.get("document_ref", ""),
            flag.get("issue_category", ""),
            flag.get("itc_at_risk_rs", 0),
            flag.get("provision", ""),
            flag.get("rule_ref", ""),
            rlvl,
            flag.get("time_sensitivity", ""),
            flag.get("recommended_action", ""),
            flag.get("compliance_note", ""),
        ]
        for c, v_val in enumerate(vals, 1):
            fmt    = "#,##0.00" if c == 5 else None
            halign = "right"    if c == 5 else "left"
            _body(ws.cell(r, c), v_val, bg=bg,
                  halign=halign, fmt=fmt)
        ws.row_dimensions[r].height = 20

    widths = [8, 26, 26, 32, 16, 36, 12, 10, 26, 52, 55]
    for c, w in enumerate(widths, 1):
        _col_w(ws, c, w)


def write_query_sheet(wb, query_data: list[dict]):
    """Write Supplier Query Draft sheet with Agent 5 results."""
    if "Supplier Query Draft" in wb.sheetnames:
        del wb["Supplier Query Draft"]
    ws = wb.create_sheet("Supplier Query Draft")
    ws.freeze_panes = "A3"
    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:F1")
    _hdr(ws["A1"],
         "SUPPLIER QUERY DRAFT  |  Agent 5: Query Letter Drafter",
         size=10)
    ws.row_dimensions[1].height = 22

    headers = [
        "Supplier Name", "Supplier GSTIN",
        "Flag Ref", "Document Ref",
        "Supplier Summary",
        "Query Paragraph (copy to email)"
    ]
    for c, h in enumerate(headers, 1):
        _hdr(ws.cell(2, c), h, bg=HDR_MID)
    ws.row_dimensions[2].height = 28

    row_bgs = [BLUE_LITE, WHITE]
    r = 3
    for sup in query_data:
        sname   = sup.get("supplier_name", "")
        sgstin  = sup.get("supplier_gstin", "")
        summary = sup.get("supplier_summary", "")
        paras   = sup.get("query_paras", [])

        for j, para in enumerate(paras):
            bg = row_bgs[(r - 3) % 2]
            vals = [
                sname if j == 0 else "",
                sgstin if j == 0 else "",
                para.get("flag_id", ""),
                para.get("document_ref", ""),
                summary if j == 0 else "",
                para.get("query_para", ""),
            ]
            for c, v_val in enumerate(vals, 1):
                cell_bg = YELLOW if c == 6 else bg
                wrap    = True   if c in (5, 6) else False
                ws.cell(r, c).value     = v_val
                ws.cell(r, c).fill      = PatternFill("solid", fgColor=cell_bg)
                ws.cell(r, c).font      = Font(name="Arial", size=9)
                ws.cell(r, c).alignment = Alignment(vertical="center",
                                                     wrap_text=wrap,
                                                     horizontal="left")
                ws.cell(r, c).border    = _BDR
            ws.row_dimensions[r].height = 60 if j == 0 else 45
            r += 1

    widths = [26, 20, 10, 26, 40, 80]
    for c, w in enumerate(widths, 1):
        _col_w(ws, c, w)


# ==============================================================
# SECTION 9 -- MAIN ORCHESTRATOR
# ==============================================================

def main(xlsx_path: str):
    print(f"\nITC RECONCILIATION INTELLIGENCE LAYER  --  Y K JONEJA & CO")
    print("=" * 72)
    print(f"  Input  : {xlsx_path}")
    print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 72)

    # ── Step 1: Read workbook ──────────────────────────────────
    print("\nSTEP 1  Reading reconciliation output")
    print("-" * 72)
    if not Path(xlsx_path).exists():
        print(f"  ERR  File not found: {xlsx_path}")
        sys.exit(1)

    sheets      = read_sheets(xlsx_path)
    client_name = sheets["meta"]["client_name"]
    period      = sheets["meta"]["return_period"]
    books_ex    = sheets["books_exceptions"]
    portal_ex   = sheets["portal_exceptions"]
    fuzzy_df    = sheets["fuzzy_review"]

    print(f"\n  Client : {client_name}")
    print(f"  Period : {period}")

    # ── Step 2: Pre-filter ─────────────────────────────────────
    print("\nSTEP 2  Running pre-filter")
    print("-" * 72)
    if books_ex.empty or portal_ex.empty:
        print("  WARN  One or both exception sheets empty — skipping pre-filter")
        candidates = []
    else:
        candidates = run_prefilter(books_ex, portal_ex)
        print(f"  Candidate pairs found : {len(candidates)}")

    candidates_json = candidates_to_json(candidates)
    fuzzy_json      = _df_to_json(fuzzy_df)

    # ── Step 3: Build and run crew ─────────────────────────────
    print("\nSTEP 3  Running CrewAI crew")
    print("-" * 72)

    crew   = build_crew(candidates_json, fuzzy_json,
                        client_name, period)
    result = crew.kickoff()

    # crew.kickoff() returns the last task output as string
    # Individual task outputs are in crew.tasks[n].output.raw_output
    raw_task2 = crew.tasks[0].output.raw_output if crew.tasks[0].output else "[]"
    raw_task3 = crew.tasks[1].output.raw_output if crew.tasks[1].output else "[]"
    raw_task4 = crew.tasks[2].output.raw_output if crew.tasks[2].output else "[]"
    raw_task5 = crew.tasks[3].output.raw_output if crew.tasks[3].output else "[]"

    verdicts    = _parse_json_response(raw_task2)
    triage      = _parse_json_response(raw_task3)
    comp_flags  = _parse_json_response(raw_task4)
    query_data  = _parse_json_response(raw_task5)

    print(f"\n  Agent 2 verdicts    : {len(verdicts)}")
    print(f"  Agent 3 triage rows : {len(triage)}")
    print(f"  Agent 4 flags       : {len(comp_flags)}")
    print(f"  Agent 5 suppliers   : {len(query_data)}")

    # ── Step 4: Write back to workbook ─────────────────────────
    print("\nSTEP 4  Writing results back to workbook")
    print("-" * 72)

    wb = openpyxl.load_workbook(xlsx_path)

    write_escape_candidates_sheet(wb, candidates, verdicts)
    print("  OK  Escape Candidates sheet written")

    write_fuzzy_triage_sheet(wb, fuzzy_df, triage)
    print("  OK  Fuzzy Match Triage sheet written")

    write_compliance_sheet(wb, comp_flags)
    print("  OK  Compliance Flags sheet written")

    write_query_sheet(wb, query_data)
    print("  OK  Supplier Query Draft sheet written")

    wb.save(xlsx_path)
    print(f"\n  Saved : {xlsx_path}")
    print(f"  Done  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 72)


# ==============================================================
# ENTRY POINT
# ==============================================================

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python itc_recon_crew.py <path_to_v9_1_output.xlsx>")
        print("       Set GOOGLE_API_KEY in environment first.")
        sys.exit(1)
    main(sys.argv[1])